# Avance 4. Modelos alternativos

## Equipo 41

**Proyecto:** Evaluación de modelos de visión computacional para la detección de eventos de ocultamiento de mercancía en video

**Integrantes:**
- Jesús Alberto Jiménez Ramos
- Nélida Silva Cabrera
- José Luis Tinajero Guerrero

**Empresa:** BLOCK NETWORKS S.A. DE C.V. (BNext)

**Asesor:** Dr. Guillermo Mota Medina

**Curso:** Proyecto Integrador  
**Maestría en Inteligencia Artificial Aplicada**  
**Tecnológico de Monterrey**

**Fecha:** octubre de 2026

## 1. Objetivo del avance

El objetivo de este notebook es evaluar modelos alternativos para la detección de eventos de ocultamiento de mercancía y determinar cuáles presentan el comportamiento más prometedor para continuar con el proyecto.

El avance parte del modelo baseline R3D-18 desarrollado en el Avance 3 y amplía la experimentación mediante arquitecturas con diferentes mecanismos de modelado espacio-temporal. Se evaluarán seis modelos individuales:

1. R3D-18.
2. MC3-18.
3. R(2+1)D-18.
4. S3D.
5. MViT-V2-S.
6. STG-NF sobre secuencias de pose.

Los primeros cinco modelos procesan información RGB y se evaluarán bajo un protocolo común de clasificación binaria. STG-NF utiliza una representación diferente, basada en puntos corporales, y formula el problema como detección de anomalías. Por esta razón, sus resultados se reportarán como un experimento complementario dentro de la comparación de modelos, evitando interpretar sus métricas como directamente equivalentes a las obtenidas por los clasificadores RGB cuando los datos y las unidades de evaluación sean diferentes.

Además de comparar los modelos, se seleccionarán los dos candidatos con mayor evidencia de desempeño y generalización dentro de condiciones experimentales comparables. Sobre estos candidatos se realizará ajuste de hiperparámetros antes de seleccionar el modelo individual que continuará hacia la siguiente etapa del proyecto.

La evaluación considera no solamente el desempeño predictivo, sino también estabilidad, errores observados, tiempo de entrenamiento, demanda computacional y pertinencia para el caso de uso planteado por BNext.

## 2. Relación entre la rúbrica y el diseño experimental

La rúbrica del Avance 4 solicita construir al menos seis modelos individuales utilizando algoritmos variados, comparar su rendimiento, seleccionar los dos modelos más prometedores, ajustar sus hiperparámetros y elegir un modelo individual final.

Debido a que este proyecto trabaja con reconocimiento de acciones en video, los criterios de la rúbrica se adaptan de la siguiente manera:

| Criterio de la rúbrica | Aplicación en este notebook |
|---|---|
| Seis modelos diferentes | Se evalúan cinco arquitecturas para video RGB y un modelo basado en secuencias de pose. Los modelos representan diferentes mecanismos para capturar información espacial y temporal. |
| Comparación de desempeño | Los modelos RGB se comparan bajo las mismas particiones y definición de clases. Average Precision (AP) se utiliza como métrica principal. Se reportan además ROC-AUC, PR-AUC, EER, precision, recall, F1-score y tiempos de entrenamiento. |
| Modelos con diferente representación | STG-NF utiliza secuencias corporales de pose y detección de anomalías. Se reporta dentro de los seis modelos evaluados, pero su comparación con RGB se interpreta considerando las diferencias de representación, datos y objetivo de aprendizaje. |
| Selección de dos modelos | La selección se realiza utilizando exclusivamente resultados de validación y considerando la métrica principal, métricas complementarias, estabilidad y costo computacional. El conjunto de prueba no se utiliza para seleccionar candidatos ni hiperparámetros. |
| Ajuste fino | Los dos modelos seleccionados se someten a experimentos de ajuste de hiperparámetros utilizando únicamente entrenamiento y validación. |
| Modelo individual final | El modelo final se selecciona considerando desempeño, errores, capacidad de generalización, complejidad y requerimientos computacionales, además de la métrica principal. |

La comparación pretende identificar diferencias reales entre enfoques y evitar atribuir mejoras a cambios en datos, particiones o procedimientos de evaluación.

## 3. Punto de partida: modelo baseline

El Avance 3 estableció R3D-18 como modelo baseline para el problema de clasificación binaria de clips de video. El modelo utiliza pesos preentrenados en Kinetics-400 y fue ajustado para distinguir las clases `ocultamiento` y `sin_ocultamiento`.

La unidad de análisis utilizada para los modelos RGB es un clip compuesto por 16 frames con resolución espacial de 112 × 112 píxeles:

$$
X_i \in \mathbb{R}^{3 \times 16 \times 112 \times 112}
$$

La variable objetivo se define como:

$$
y_i =
\begin{cases}
0, & \text{sin\_ocultamiento} \\
1, & \text{ocultamiento}
\end{cases}
$$


El Avance 3 también permitió identificar casos donde las etiquetas originales asociadas con `shoplifting` no correspondían con suficiente claridad a la definición operacional de ocultamiento utilizada por el proyecto. Estos casos se analizaron cualitativamente y constituyen el punto de partida para una revisión dirigida del ground truth antes de realizar la comparación de modelos del presente avance.

## 4. Experimento 0: calidad del ground truth

Antes de evaluar modelos alternativos se realiza un experimento dirigido a analizar el efecto de la calidad de las etiquetas sobre las métricas del baseline.

Durante el análisis de errores del Avance 3 se revisaron los 13 falsos negativos obtenidos por R3D-18 sobre el conjunto de prueba. Once de ellos fueron clasificados como ambiguos respecto a la definición operacional de `ocultamiento`, mientras que dos presentaron evidencia suficiente para conservar la etiqueta positiva.

La revisión no constituye una nueva auditoría completa del dataset. Su alcance se limita a los casos que ya fueron identificados y revisados durante el análisis cualitativo del Avance 3.

Los clips ambiguos:

- permanecen documentados para conservar trazabilidad;
- no se reasignan automáticamente a `sin_ocultamiento`;
- se excluyen del conjunto utilizado para calcular las métricas binarias del Avance 4.

Los clips cuya etiqueta fue considerada clara conservan su clase original.

Las particiones de entrenamiento, validación y prueba no se redistribuyen después de esta revisión. Por lo tanto, los grupos y videos mantienen la partición asignada previamente.

El conjunto resultante se denomina en este notebook **dataset v2**. Una vez aplicada la revisión dirigida, sus etiquetas y particiones quedan congeladas para todos los experimentos posteriores.

El Experimento 0 compara las métricas del mismo checkpoint R3D-18 utilizando:

1. El ground truth empleado en el Avance 3.
2. El ground truth revisado utilizado en el Avance 4.

Cualquier diferencia entre ambas evaluaciones representa un cambio en la estimación del desempeño debido a la calidad y alineación semántica del ground truth. No representa una mejora del modelo, ya que los parámetros y predicciones de R3D-18 permanecen sin cambios.

## 5. Definiciones y decisiones de modelado

### 5.1 Definición operacional de la clase positiva

Se considera `ocultamiento` cuando el clip contiene evidencia visual suficiente de que una persona aproxima o introduce mercancía hacia una zona de ocultamiento definida para el proyecto, como ropa, bolsillo, bolsa o mochila.

La clase `sin_ocultamiento` incluye situaciones en las que la mercancía permanece visible o donde no ocurre la acción objetivo.

Los clips cuya acción no puede determinarse con suficiente claridad se consideran ambiguos y quedan fuera de la evaluación binaria principal.

### 5.2 Unidad de análisis para modelos RGB

Los modelos RGB procesan clips temporales en lugar de videos completos. Esta decisión permite asociar la predicción con el segmento donde puede observarse la acción objetivo y evita transferir automáticamente la etiqueta general de un video a todos sus fragmentos.

La configuración inicial utiliza 16 frames por clip. Las transformaciones espaciales requeridas por cada arquitectura se documentarán cuando difieran del preprocesamiento utilizado por R3D-18.

### 5.3 Unidad de análisis para poses

STG-NF utiliza ventanas temporales de puntos corporales correspondientes a una persona. La representación de RetailS utiliza puntos corporales en formato COCO17 e incluye coordenadas y valores de confianza.

Este enfoque modela principalmente patrones de movimiento corporal y no dispone directamente de información visual detallada sobre el producto, textura, vestimenta o destino final de la mercancía.

### 5.4 Tipo de aprendizaje

Los modelos RGB se formulan como clasificadores supervisados binarios.

STG-NF se utiliza como modelo de detección de anomalías sobre secuencias de pose. Aprende patrones asociados con actividad normal y genera una puntuación de anomalía para las secuencias evaluadas.

La puntuación de STG-NF no se interpreta directamente como una probabilidad de ocultamiento.

### 5.5 Transferencia de aprendizaje

Cuando se disponga de pesos preentrenados compatibles, se utilizarán como punto de partida para reducir el costo de entrenamiento y aprovechar representaciones aprendidas sobre conjuntos de video de mayor escala.

Las fuentes de los pesos, transformaciones requeridas y capas modificadas para cada arquitectura se documentarán en la sección correspondiente a cada experimento.

## 6. Criterios de calidad experimental

Los experimentos se diseñan para que las diferencias observadas entre modelos puedan atribuirse principalmente al enfoque de modelado y no a cambios inadvertidos en los datos o en el procedimiento de evaluación.

### 6.1 Calidad de las etiquetas

Las etiquetas utilizadas deben corresponder con la definición operacional de `ocultamiento` adoptada por el proyecto. Los casos ambiguos identificados durante la revisión del Avance 3 se excluyen de la evaluación binaria del dataset v2.

### 6.2 Independencia entre particiones

Todos los clips derivados de un mismo video o grupo relacionado permanecen dentro de una sola partición. No se redistribuyen ejemplos después de observar el desempeño de los modelos.

### 6.3 Separación entre selección y evaluación final

El conjunto de entrenamiento se utiliza para optimizar los parámetros del modelo.

El conjunto de validación se utiliza para:

- seleccionar checkpoints;
- comparar configuraciones;
- seleccionar los dos modelos candidatos;
- ajustar hiperparámetros;
- definir el umbral de decisión cuando corresponda.

El conjunto de prueba se reserva para estimar el desempeño de configuraciones ya seleccionadas.

### 6.4 Reproducibilidad

Cada experimento deberá registrar, cuando aplique:

- Arquitectura.
- Inicialización y pesos preentrenados.
- Semilla aleatoria.
- Tamaño de lote.
- Optimizador.
- Tasa de aprendizaje.
- Regularización.
- Número máximo de épocas.
- Criterio de early stopping.
- Checkpoint seleccionado.
- Tiempo de entrenamiento.
- Dispositivo utilizado.

### 6.5 Comparabilidad

Los cinco modelos RGB utilizan las mismas particiones y la misma definición de las clases.

Cuando una arquitectura requiera una resolución, normalización u otra transformación específica, la diferencia se documentará explícitamente y no se interpretará como un cambio en el ground truth.

STG-NF se analiza mediante un protocolo separado porque utiliza otra representación y otro conjunto de datos.

### 6.6 Control del conjunto de prueba

Después de congelar el dataset v2, las etiquetas de prueba no se modificarán en función de los errores producidos por los modelos del Avance 4.

Los errores posteriores podrán analizarse cualitativamente, pero dicho análisis no se utilizará para alterar el ground truth de esta ronda experimental.

## 7. Métricas de evaluación

Debido al desbalance entre `ocultamiento` y `sin_ocultamiento`, la evaluación no se basa únicamente en accuracy.

### 7.1 Métrica principal

**Average Precision (AP)** se utiliza como métrica principal para la selección y comparación de los modelos RGB.

AP resume el comportamiento precision-recall a través de diferentes umbrales y mantiene continuidad con el criterio utilizado para seleccionar el checkpoint del baseline en el Avance 3.

### 7.2 Métricas complementarias

Se reportarán también:

- PR-AUC.
- ROC-AUC.
- Equal Error Rate (EER).
- Precision.
- Recall.
- F1-score.
- Specificity.
- Accuracy.
- Matriz de confusión.

Las métricas dependientes de un umbral se calcularán utilizando un umbral seleccionado exclusivamente a partir del conjunto de validación.

### 7.3 Métricas computacionales

La rúbrica solicita registrar tiempos de entrenamiento. Adicionalmente, cuando sea posible, se documentarán:

- Tiempo total de entrenamiento.
- Tiempo medio por época.
- Throughput o tiempo de inferencia.
- Memoria GPU observada.
- Número aproximado de parámetros.

Estas mediciones permitirán complementar la selección del modelo con información sobre costo computacional y posible viabilidad para una implementación futura.

## 8. Protocolos experimentales

El Avance 4 organiza la experimentación en etapas consecutivas que permiten revisar primero la calidad de la referencia de evaluación, preparar posteriormente los experimentos comparables y finalmente seleccionar los modelos que continuarán hacia el ajuste de hiperparámetros.

El protocolo contempla cuatro etapas:

1. análisis de sensibilidad de las métricas a la calidad del ground truth
2. evaluación comparativa de arquitecturas RGB
3. evaluación complementaria de un modelo basado en pose
4. selección de dos candidatos para ajuste fino

Las decisiones de selección de modelos y configuración de hiperparámetros utilizarán exclusivamente información de entrenamiento y validación. El conjunto de prueba se mantendrá reservado para la evaluación final correspondiente.

In [ ]:
# ============================================================
# Configuración general del Avance 4
# ============================================================

SEED = 42

PRIMARY_METRIC = "average_precision"

RGB_MODELS = {
    "RGB-0": "R3D-18",
    "RGB-1": "MC3-18",
    "RGB-2": "R(2+1)D-18",
    "RGB-3": "S3D",
    "RGB-4": "MViT-V2-S",
}

POSE_MODEL = {
    "POSE-0": "STG-NF"
}

AMBIGUOUS_TEST_CLIPS_AVANCE3 = [
    # DCSASS
    "Shoplifting006_x264_28",
    "Shoplifting006_x264_29",
    "Shoplifting006_x264_19",
    "Shoplifting021_x264_23",

    # MNNIT
    "MNNIT_065",
    "MNNIT_067",
    "MNNIT_013",
    "MNNIT_064",
    "MNNIT_069",
    "MNNIT_066",
    "MNNIT_068",
]

print("=== CONFIGURACIÓN DEL AVANCE 4 ===")
print(f"Semilla: {SEED}")
print(f"Métrica principal: {PRIMARY_METRIC}")
print(f"Modelos RGB: {len(RGB_MODELS)}")
print(f"Modelo de poses: {list(POSE_MODEL.values())[0]}")
print(f"Total de modelos: {len(RGB_MODELS) + len(POSE_MODEL)}")
print(
    "Clips ambiguos identificados en el Avance 3:",
    len(AMBIGUOUS_TEST_CLIPS_AVANCE3)
)

### 8.1 Experimento 0: sensibilidad de las métricas a la calidad del ground truth

Se reutiliza el checkpoint R3D-18 del Avance 3 y se comparan sus métricas utilizando el ground truth original y el ground truth revisado.

No se realiza entrenamiento adicional.

Este experimento permite cuantificar cuánto cambia la evaluación del baseline cuando los casos ambiguos previamente identificados se excluyen de la tarea binaria.

La revisión se limita a los 11 casos ambiguos identificados durante el análisis de falsos negativos del Avance 3. El objetivo es evaluar el efecto de esta decisión sobre las métricas sin modificar las predicciones generadas por el modelo original.

In [ ]:
from pathlib import Path
import os
import pandas as pd


def find_repo_root():
    """
    Localiza la raíz del repositorio sin depender de rutas
    personales específicas del equipo.
    """

    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
    ]

    repo_env = os.environ.get("MNA_REPO_DIR")

    if repo_env:
        candidates.append(Path(repo_env))

    candidates.append(
        Path("/opt/workspace/MNA---Proyecto-Integrador")
    )

    for candidate in candidates:
        data_dir = (
            candidate
            / "Datos"
            / "Preparacion_Datasets"
        )

        if data_dir.is_dir():
            return candidate

    if not TEST_PREDICTIONS_PATH.exists():
        raise FileNotFoundError(
            "No se encontró el archivo esperado: "
            "Notebooks/checkpoints/"
            "r3d18_baseline_test_predictions.csv"
        )


REPO_DIR = find_repo_root()

CHECKPOINT_DIR = (
    REPO_DIR
    / "Notebooks"
    / "checkpoints"
)

TEST_PREDICTIONS_PATH = (
    CHECKPOINT_DIR
    / "r3d18_baseline_test_predictions.csv"
)

if not TEST_PREDICTIONS_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo: {TEST_PREDICTIONS_PATH}"
    )

df_test_v1 = pd.read_csv(
    TEST_PREDICTIONS_PATH
)

print("=== PREDICCIONES BASELINE CARGADAS ===")
print("Archivo:", TEST_PREDICTIONS_PATH.relative_to(REPO_DIR))
print("Registros:", len(df_test_v1))
print(
    "Positivos reales:",
    int(df_test_v1["clase_binaria"].sum())
)
print(
    "Negativos reales:",
    int(
        len(df_test_v1)
        - df_test_v1["clase_binaria"].sum()
    )
)
print(
    "Split:",
    df_test_v1["split"].unique().tolist()
)

In [ ]:
# ============================================================
# Validación de clips ambiguos identificados en el Avance 3
# ============================================================

df_ambiguos_v1 = (
    df_test_v1[
        df_test_v1["clip_id"].isin(
            AMBIGUOUS_TEST_CLIPS_AVANCE3
        )
    ]
    .copy()
    .sort_values(
        ["dataset", "video_id", "clip_id"]
    )
    .reset_index(drop=True)
)

clips_encontrados = set(
    df_ambiguos_v1["clip_id"]
)

clips_esperados = set(
    AMBIGUOUS_TEST_CLIPS_AVANCE3
)

clips_faltantes = (
    clips_esperados
    - clips_encontrados
)

duplicados = (
    df_ambiguos_v1["clip_id"]
    .duplicated()
    .sum()
)

todos_positivos = (
    df_ambiguos_v1["clase_binaria"]
    .eq(1)
    .all()
)

print("=== VALIDACIÓN DE CLIPS AMBIGUOS ===")
print(
    "Clips ambiguos esperados:",
    len(clips_esperados)
)
print(
    "Clips ambiguos encontrados:",
    len(df_ambiguos_v1)
)
print(
    "Clips faltantes:",
    len(clips_faltantes)
)
print(
    "Registros duplicados:",
    int(duplicados)
)
print(
    "Todos tienen etiqueta positiva:",
    todos_positivos
)

assert len(df_ambiguos_v1) == 11
assert len(clips_faltantes) == 0
assert duplicados == 0
assert todos_positivos

display(
    df_ambiguos_v1[
        [
            "dataset",
            "video_id",
            "clip_id",
            "clase_binaria",
            "p_ocultamiento",
            "pred_threshold_final"
        ]
    ]
)

In [ ]:
# ============================================================
# Construcción del conjunto de prueba v2
# ============================================================

df_test_v2 = (
    df_test_v1[
        ~df_test_v1["clip_id"].isin(
            AMBIGUOUS_TEST_CLIPS_AVANCE3
        )
    ]
    .copy()
    .reset_index(drop=True)
)

n_v1 = len(df_test_v1)
n_v2 = len(df_test_v2)

positivos_v1 = int(
    df_test_v1["clase_binaria"].sum()
)

positivos_v2 = int(
    df_test_v2["clase_binaria"].sum()
)

negativos_v1 = n_v1 - positivos_v1
negativos_v2 = n_v2 - positivos_v2

ambiguos_restantes = (
    df_test_v2["clip_id"]
    .isin(AMBIGUOUS_TEST_CLIPS_AVANCE3)
    .sum()
)

print("=== CONSTRUCCIÓN DE TEST V2 ===")
print("Registros test v1:", n_v1)
print("Registros test v2:", n_v2)
print("Clips excluidos:", n_v1 - n_v2)
print()
print("Positivos test v1:", positivos_v1)
print("Positivos test v2:", positivos_v2)
print()
print("Negativos test v1:", negativos_v1)
print("Negativos test v2:", negativos_v2)
print()
print(
    "Clips ambiguos restantes en test v2:",
    int(ambiguos_restantes)
)

assert n_v1 == 147
assert n_v2 == 136
assert n_v1 - n_v2 == 11

assert positivos_v1 == 28
assert positivos_v2 == 17

assert negativos_v1 == 119
assert negativos_v2 == 119

assert ambiguos_restantes == 0

In [ ]:
# ============================================================
# Experimento 0
# Comparación de métricas entre test v1 y test v2
# ============================================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    auc,
    roc_curve
)


THRESHOLD_FINAL = 0.7090


def calcular_metricas(
    y_true,
    y_prob,
    threshold=0.5
):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    average_precision = average_precision_score(
        y_true,
        y_prob
    )

    precision_curve, recall_curve, _ = (
        precision_recall_curve(
            y_true,
            y_prob
        )
    )

    pr_auc = auc(
        recall_curve,
        precision_curve
    )

    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_prob
    )

    fnr = 1 - tpr

    idx_eer = np.nanargmin(
        np.abs(fpr - fnr)
    )

    eer = (
        fpr[idx_eer]
        + fnr[idx_eer]
    ) / 2

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc_auc,
        "average_precision": average_precision,
        "pr_auc": pr_auc,
        "eer": eer,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


metricas_v1 = calcular_metricas(
    df_test_v1["clase_binaria"],
    df_test_v1["p_ocultamiento"],
    threshold=THRESHOLD_FINAL
)

metricas_v2 = calcular_metricas(
    df_test_v2["clase_binaria"],
    df_test_v2["p_ocultamiento"],
    threshold=THRESHOLD_FINAL
)


metricas_mostrar = [
    "average_precision",
    "pr_auc",
    "roc_auc",
    "eer",
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "tn",
    "fp",
    "fn",
    "tp"
]


comparacion_exp0 = pd.DataFrame(
    {
        "test_v1": [
            metricas_v1[m]
            for m in metricas_mostrar
        ],
        "test_v2": [
            metricas_v2[m]
            for m in metricas_mostrar
        ]
    },
    index=metricas_mostrar
)

comparacion_exp0["delta"] = (
    comparacion_exp0["test_v2"]
    - comparacion_exp0["test_v1"]
)

display(
    comparacion_exp0.round(4)
)

#### 8.1.1 Resultados del Experimento 0

La evaluación con el ground truth original reproduce las métricas obtenidas en el Avance 3, lo que confirma que las predicciones congeladas del baseline y el procedimiento de cálculo se conservaron correctamente.

Después de excluir los 11 clips previamente clasificados como ambiguos para la definición operacional de `ocultamiento`, el conjunto de prueba pasa de 147 a 136 observaciones. Los 119 ejemplos negativos permanecen sin cambios, mientras que los positivos disminuyen de 28 a 17.

La modificación del ground truth produce cambios relevantes en las métricas del mismo checkpoint R3D-18. Average Precision aumenta de 0.5393 a 0.6067, PR-AUC de 0.5244 a 0.5858 y ROC-AUC de 0.8472 a 0.9249. EER disminuye de 0.2164 a 0.1092.

En las métricas dependientes del umbral seleccionado previamente en validación, recall aumenta de 0.5357 a 0.8824 y F1-score de 0.5556 a 0.6977. Precision permanece en 0.5769 y specificity en 0.9076.

Este comportamiento se explica por la naturaleza de los casos excluidos. Los 11 clips eran positivos que el baseline había clasificado como negativos. Al retirarlos de la evaluación, los falsos negativos disminuyen de 13 a 2, mientras que TP, FP y TN permanecen sin cambios. Por esta razón, recall aumenta de manera considerable, mientras que precision y specificity permanecen iguales.

Los resultados no representan una mejora del modelo. El checkpoint, sus parámetros y sus probabilidades permanecen sin cambios. La diferencia corresponde al efecto de evaluar el mismo modelo con una versión del ground truth más alineada con la definición operacional del proyecto.

Esta revisión tiene un alcance limitado. Los clips excluidos fueron identificados mediante el análisis cualitativo de falsos negativos realizado en el Avance 3 y no mediante una nueva auditoría exhaustiva de todo el conjunto de prueba. Por esta razón, el experimento se interpreta como un análisis de sensibilidad de las métricas ante problemas de alineación semántica de las etiquetas y no como una validación completa de la calidad del ground truth.

A partir de este punto, el conjunto de evaluación revisado queda congelado y no se modificarán sus etiquetas en función de los errores producidos por los modelos alternativos del Avance 4.

### 8.1.2 Dataset v2 y particiones de modelado

A partir de los resultados del Experimento 0 se construye una versión congelada del conjunto de datos que será utilizada en los experimentos posteriores del Avance 4.

Los 11 clips identificados previamente como ambiguos se conservan en el registro original, pero se marcan como no elegibles para la evaluación binaria de `ocultamiento`. No se reasignan a la clase negativa.

La nueva versión conserva sin cambios las particiones de entrenamiento y validación. La modificación afecta únicamente al conjunto de prueba.

Para asegurar reproducibilidad se guarda la versión revisada del split y se calcula su identificador SHA-256. Posteriormente se verifican la distribución de clases y la independencia de los grupos experimentales entre `train`, `validation` y `test`.

In [ ]:
# ============================================================
# Congelación del dataset v2 para el Avance 4
# ============================================================

import hashlib


BASE_DIR = (
    REPO_DIR
    / "Datos"
    / "Preparacion_Datasets"
)

SPLIT_V1_PATH = (
    BASE_DIR
    / "baseline_split_equipo41.csv"
)

SPLIT_V2_PATH = (
    BASE_DIR
    / "avance4_split_ground_truth_v2.csv"
)


if not SPLIT_V1_PATH.exists():
    raise FileNotFoundError(
        "No se encontró el archivo esperado: "
        "Datos/Preparacion_Datasets/"
        "baseline_split_equipo41.csv"
    )


df_split_v2 = pd.read_csv(
    SPLIT_V1_PATH
)

assert len(df_split_v2) == 985
assert df_split_v2["clip_id"].notna().all()
assert df_split_v2["split"].notna().all()


mascara_ambiguos = (
    df_split_v2["clip_id"]
    .isin(AMBIGUOUS_TEST_CLIPS_AVANCE3)
)

assert mascara_ambiguos.sum() == 11

assert (
    df_split_v2.loc[
        mascara_ambiguos,
        "split"
    ]
    .eq("test")
    .all()
)

assert (
    df_split_v2.loc[
        mascara_ambiguos,
        "clase_binaria"
    ]
    .eq(1)
    .all()
)


df_split_v2["revision_etiqueta"] = (
    "sin_revision"
)

df_split_v2["elegible_avance4"] = True


df_split_v2.loc[
    mascara_ambiguos,
    "revision_etiqueta"
] = "ambigua_para_ocultamiento"

df_split_v2.loc[
    mascara_ambiguos,
    "elegible_avance4"
] = False


if SPLIT_V2_PATH.exists():

    df_split_v2_existente = pd.read_csv(
        SPLIT_V2_PATH
    )

    pd.testing.assert_frame_equal(
        df_split_v2,
        df_split_v2_existente,
        check_dtype=False
    )

    estado_archivo = (
        "El archivo ya existía y coincide con el dataset v2."
    )

else:

    df_split_v2.to_csv(
        SPLIT_V2_PATH,
        index=False
    )

    estado_archivo = (
        "Dataset v2 guardado correctamente."
    )


def calcular_sha256(ruta):
    sha256 = hashlib.sha256()

    with open(ruta, "rb") as archivo:
        for bloque in iter(
            lambda: archivo.read(8192),
            b""
        ):
            sha256.update(bloque)

    return sha256.hexdigest()


sha256_v2 = calcular_sha256(
    SPLIT_V2_PATH
)


resumen_elegibles = (
    df_split_v2[
        df_split_v2["elegible_avance4"]
    ]
    ["split"]
    .value_counts()
    .reindex(
        ["train", "val", "test"]
    )
)


print("=== DATASET V2 CONGELADO ===")
print(estado_archivo)
print(
    "Archivo:",
    SPLIT_V2_PATH.relative_to(REPO_DIR)
)
print(
    "Registros totales:",
    len(df_split_v2)
)
print(
    "Casos ambiguos excluidos:",
    int((~df_split_v2["elegible_avance4"]).sum())
)

print("\nRegistros elegibles por partición:")
print(resumen_elegibles)

print("\nSHA-256:")
print(sha256_v2)

In [ ]:
# ============================================================
# Particiones elegibles para los experimentos RGB
# ============================================================

df_modelado_v2 = (
    df_split_v2[
        df_split_v2["elegible_avance4"]
    ]
    .copy()
    .reset_index(drop=True)
)

df_train_v2 = (
    df_modelado_v2[
        df_modelado_v2["split"] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)

df_val_v2 = (
    df_modelado_v2[
        df_modelado_v2["split"] == "val"
    ]
    .copy()
    .reset_index(drop=True)
)

df_test_modelado_v2 = (
    df_modelado_v2[
        df_modelado_v2["split"] == "test"
    ]
    .copy()
    .reset_index(drop=True)
)


def resumen_particion(nombre, df):
    positivos = int(
        df["clase_binaria"].sum()
    )

    negativos = (
        len(df)
        - positivos
    )

    prevalencia = (
        positivos / len(df)
    )

    return {
        "particion": nombre,
        "n": len(df),
        "negativos": negativos,
        "positivos": positivos,
        "prevalencia_ocultamiento": prevalencia
    }


resumen_particiones_v2 = pd.DataFrame(
    [
        resumen_particion(
            "train",
            df_train_v2
        ),
        resumen_particion(
            "val",
            df_val_v2
        ),
        resumen_particion(
            "test",
            df_test_modelado_v2
        )
    ]
)

assert len(df_train_v2) == 702
assert len(df_val_v2) == 136
assert len(df_test_modelado_v2) == 136

assert (
    df_modelado_v2["revision_etiqueta"]
    .ne("ambigua_para_ocultamiento")
    .all()
)

print("=== PARTICIONES DE MODELADO V2 ===")

display(
    resumen_particiones_v2.style.format(
        {
            "prevalencia_ocultamiento": "{:.4f}"
        }
    )
)

In [ ]:
# ============================================================
# Verificación de independencia entre particiones
# ============================================================

grupos_por_split = (
    df_modelado_v2
    .groupby("grupo_experimental")["split"]
    .nunique()
)

grupos_con_fuga = (
    grupos_por_split[
        grupos_por_split > 1
    ]
)

grupos_train = set(
    df_train_v2["grupo_experimental"]
)

grupos_val = set(
    df_val_v2["grupo_experimental"]
)

grupos_test = set(
    df_test_modelado_v2["grupo_experimental"]
)

interseccion_train_val = (
    grupos_train
    & grupos_val
)

interseccion_train_test = (
    grupos_train
    & grupos_test
)

interseccion_val_test = (
    grupos_val
    & grupos_test
)

clips_duplicados = (
    df_modelado_v2["clip_id"]
    .duplicated()
    .sum()
)

print("=== VERIFICACIÓN DE INDEPENDENCIA ===")

print(
    "Grupos experimentales totales:",
    df_modelado_v2["grupo_experimental"].nunique()
)

print(
    "Grupos en train:",
    len(grupos_train)
)

print(
    "Grupos en val:",
    len(grupos_val)
)

print(
    "Grupos en test:",
    len(grupos_test)
)

print()

print(
    "Grupos compartidos train / val:",
    len(interseccion_train_val)
)

print(
    "Grupos compartidos train / test:",
    len(interseccion_train_test)
)

print(
    "Grupos compartidos val / test:",
    len(interseccion_val_test)
)

print()

print(
    "Grupos asignados a más de un split:",
    len(grupos_con_fuga)
)

print(
    "Clip IDs duplicados:",
    int(clips_duplicados)
)

assert len(grupos_con_fuga) == 0
assert len(interseccion_train_val) == 0
assert len(interseccion_train_test) == 0
assert len(interseccion_val_test) == 0
assert clips_duplicados == 0

#### Conclusiones del dataset v2 y las particiones de modelado

La revisión del ground truth produce una versión congelada del dataset para los experimentos posteriores del Avance 4. Los 11 casos ambiguos permanecen registrados en el archivo, pero se excluyen de la tarea binaria mediante el campo `elegible_avance4`.

Las particiones elegibles quedan distribuidas de la siguiente forma:

| Partición | Total | Sin ocultamiento | Ocultamiento | Prevalencia de ocultamiento |
|---|---:|---:|---:|---:|
| Train | 702 | 570 | 132 | 0.1880 |
| Validation | 136 | 106 | 30 | 0.2206 |
| Test | 136 | 119 | 17 | 0.1250 |

Train y validation permanecen sin cambios respecto al Avance 3. La modificación se concentra en test, donde los 11 casos excluidos correspondían originalmente a la clase positiva.

La menor prevalencia de `ocultamiento` en test debe considerarse al interpretar métricas Precision-Recall. Para `test_v2`, la referencia aproximada de Average Precision de un clasificador no informativo es 0.1250. Por esta razón, accuracy no se utilizará como criterio principal para seleccionar modelos.

También se verificó la independencia de las particiones. Los 63 grupos experimentales se distribuyen en 46 grupos de entrenamiento, 9 de validación y 8 de prueba. No existen grupos compartidos entre particiones, grupos asignados a más de un split ni identificadores de clip duplicados.

La versión congelada queda identificada mediante su SHA-256:

`b85d6b55e8c32420a4df5a0fb9a10e558105beee47f1d4a98ad83d63fcbda21e`

A partir de este punto, todos los modelos RGB utilizarán exactamente estas particiones. Train y validation se emplearán para entrenamiento, selección de checkpoints y ajuste de hiperparámetros. Test permanecerá reservado hasta que la configuración correspondiente haya sido fijada.

### 8.2 Preparación común de los experimentos RGB

La comparación inicial considera cinco arquitecturas para reconocimiento de acciones directamente sobre video RGB.

| ID | Arquitectura | Familia |
|---|---|---|
| RGB-0 | R3D-18 | Convolución 3D |
| RGB-1 | MC3-18 | Convolución 3D/2D mixta |
| RGB-2 | R(2+1)D-18 | Convolución espacial y temporal factorizada |
| RGB-3 | S3D | Convolución 3D separable |
| RGB-4 | MViT-V2-S | Transformer multiescala para video |

R3D-18 funciona como arquitectura de referencia debido a su uso previo como baseline. Los otros cuatro modelos permiten evaluar mecanismos diferentes para representar la información espacial y temporal de los clips.

Los cinco modelos utilizarán las mismas particiones del dataset v2, el mismo muestreo temporal de 16 frames y un protocolo común de entrenamiento y evaluación.

Antes de iniciar los entrenamientos se verificará de forma común:

1. entorno experimental y disponibilidad de los clips
2. preprocesamiento requerido por cada arquitectura
3. compatibilidad de las formas de entrada
4. implementación del dataset y los DataLoader
5. adaptación de los clasificadores al problema binario
6. política inicial de fine-tuning
7. capacidad de la GPU y tamaño de batch
8. criterios comunes de entrenamiento, evaluación y registro

La preparación común busca que las diferencias observadas posteriormente puedan atribuirse principalmente a las arquitecturas evaluadas y no a cambios en las particiones, el muestreo temporal o el procedimiento experimental.

#### 8.2.1 Entorno experimental y disponibilidad de los datos

Antes de configurar las arquitecturas se verifica que el entorno de ejecución y los datos físicos necesarios para los experimentos estén disponibles.

La configuración reproducible fija la semilla aleatoria utilizada durante los experimentos y registra las versiones principales del entorno, la disponibilidad de CUDA, el dispositivo seleccionado y la memoria total de la GPU.

También se comprueba que la variable de entorno utilizada para localizar los clips esté configurada sin incorporar rutas locales específicas dentro del notebook.

Finalmente, se valida que los clips requeridos por el dataset v2 existan físicamente antes de construir los datasets y DataLoader utilizados durante el entrenamiento.

In [ ]:
# ============================================================
# Configuración reproducible del entorno experimental
# ============================================================

import random
import numpy as np
import torch


SEED = 42


def fijar_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


fijar_semilla(SEED)


if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    nombre_dispositivo = torch.cuda.get_device_name(0)
    memoria_total_gb = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )
else:
    DEVICE = torch.device("cpu")
    nombre_dispositivo = "CPU"
    memoria_total_gb = np.nan


torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


print("=== ENTORNO EXPERIMENTAL ===")
print("Semilla:", SEED)
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Dispositivo:", DEVICE)
print("Hardware:", nombre_dispositivo)

if torch.cuda.is_available():
    print(
        "Memoria GPU total:",
        f"{memoria_total_gb:.2f} GB"
    )

print(
    "cuDNN deterministic:",
    torch.backends.cudnn.deterministic
)

print(
    "cuDNN benchmark:",
    torch.backends.cudnn.benchmark
)

In [ ]:
# ============================================================
# Verificación del dataset físico
# ============================================================

bnext_path = os.environ.get(
    "BNEXT_CLIPS_DIR"
)

if bnext_path is None:
    raise RuntimeError(
        "La variable de entorno BNEXT_CLIPS_DIR "
        "no está configurada."
    )

CLIPS_DIR = Path(
    bnext_path
)

if not CLIPS_DIR.is_dir():
    raise RuntimeError(
        "El directorio configurado en BNEXT_CLIPS_DIR "
        "no está disponible."
    )


archivos_mp4 = {
    archivo.name
    for archivo in CLIPS_DIR.glob("*.mp4")
}

clips_requeridos = set(
    df_split_v2["nombre_clip"]
)

clips_faltantes = (
    clips_requeridos
    - archivos_mp4
)


print("=== VERIFICACIÓN DEL DATASET FÍSICO ===")

print(
    "BNEXT_CLIPS_DIR configurado:",
    True
)

print(
    "Archivos MP4 disponibles:",
    len(archivos_mp4)
)

print(
    "Clips requeridos por dataset v2:",
    len(clips_requeridos)
)

print(
    "Clips faltantes:",
    len(clips_faltantes)
)


assert len(clips_requeridos) == 985
assert len(clips_faltantes) == 0

##### Conclusiones de 8.2.1

El entorno experimental quedó validado para continuar con la comparación de arquitecturas RGB.

La ejecución utiliza una semilla fija de 42, PyTorch con soporte CUDA y una GPU NVIDIA GeForce RTX 3050 Laptop con 6 GB de memoria. La configuración de cuDNN mantiene comportamiento determinista y desactiva `benchmark` para favorecer la reproducibilidad de los experimentos.

También se verificó la disponibilidad física de los datos. El dataset v2 requiere 985 clips y todos se encuentran disponibles en el directorio configurado mediante la variable de entorno correspondiente. No se detectaron archivos faltantes.

Con estas verificaciones se descartan, antes del entrenamiento, dos fuentes básicas de error experimental: diferencias no controladas en el entorno de ejecución y ausencia de clips requeridos por las particiones definidas.

#### 8.2.2 Preprocesamiento y compatibilidad de arquitecturas

Las cinco arquitecturas RGB utilizan pesos preentrenados sobre Kinetics-400, pero sus requerimientos de preprocesamiento espacial no son idénticos.

Antes de construir el dataset común se verifica que las arquitecturas, sus pesos preentrenados y las transformaciones correspondientes estén disponibles en la versión de Torchvision utilizada por el proyecto.

La comparación mantiene constantes las observaciones, las etiquetas, las particiones experimentales y el muestreo temporal de 16 frames por clip. El preprocesamiento espacial se adapta a los requerimientos asociados con los pesos preentrenados de cada arquitectura.

| Modelo | Frames | Resolución de entrada | Media | Desviación estándar |
|---|---:|---:|---|---|
| R3D-18 | 16 | 112 × 112 | `[0.43216, 0.394666, 0.37645]` | `[0.22803, 0.22145, 0.216989]` |
| MC3-18 | 16 | 112 × 112 | `[0.43216, 0.394666, 0.37645]` | `[0.22803, 0.22145, 0.216989]` |
| R(2+1)D-18 | 16 | 112 × 112 | `[0.43216, 0.394666, 0.37645]` | `[0.22803, 0.22145, 0.216989]` |
| S3D | 16 | 224 × 224 | `[0.43216, 0.394666, 0.37645]` | `[0.22803, 0.22145, 0.216989]` |
| MViT-V2-S | 16 | 224 × 224 | `[0.45, 0.45, 0.45]` | `[0.225, 0.225, 0.225]` |

La diferencia de resolución deberá considerarse al interpretar el costo computacional, ya que S3D y MViT-V2-S procesan una cantidad mayor de píxeles por frame.

Además de verificar las transformaciones, se comprobará empíricamente que las cinco arquitecturas acepten tensores con 16 frames y las dimensiones espaciales correspondientes antes de adaptar sus clasificadores al problema binario.

In [ ]:
# ============================================================
# Verificación de arquitecturas RGB disponibles
# ============================================================

import torchvision
import torchvision.models.video as video_models


modelos_verificar = {
    "R3D-18": (
        "r3d_18",
        "R3D_18_Weights"
    ),
    "MC3-18": (
        "mc3_18",
        "MC3_18_Weights"
    ),
    "R(2+1)D-18": (
        "r2plus1d_18",
        "R2Plus1D_18_Weights"
    ),
    "S3D": (
        "s3d",
        "S3D_Weights"
    ),
    "MViT-V2-S": (
        "mvit_v2_s",
        "MViT_V2_S_Weights"
    )
}


print("=== DISPONIBILIDAD DE MODELOS RGB ===")
print("Torchvision:", torchvision.__version__)
print()


for nombre, (
    nombre_modelo,
    nombre_weights
) in modelos_verificar.items():

    modelo_disponible = hasattr(
        video_models,
        nombre_modelo
    )

    weights_disponibles = hasattr(
        video_models,
        nombre_weights
    )

    print(nombre)
    print(
        "  Arquitectura disponible:",
        modelo_disponible
    )
    print(
        "  Pesos disponibles:",
        weights_disponibles
    )

    if (
        modelo_disponible
        and weights_disponibles
    ):
        clase_weights = getattr(
            video_models,
            nombre_weights
        )

        weights_default = (
            clase_weights.DEFAULT
        )

        print(
            "  Pesos por defecto:",
            weights_default
        )

        print(
            "  Transformación:"
        )

        print(
            weights_default.transforms()
        )

    print()

In [ ]:
# ============================================================
# Verificación de compatibilidad de entrada
# ============================================================

import gc
import torch

from torchvision.models.video import (
    r3d_18,
    mc3_18,
    r2plus1d_18,
    s3d,
    mvit_v2_s
)


pruebas_entrada = [
    {
        "nombre": "R3D-18",
        "constructor": r3d_18,
        "shape": (1, 3, 16, 112, 112)
    },
    {
        "nombre": "MC3-18",
        "constructor": mc3_18,
        "shape": (1, 3, 16, 112, 112)
    },
    {
        "nombre": "R(2+1)D-18",
        "constructor": r2plus1d_18,
        "shape": (1, 3, 16, 112, 112)
    },
    {
        "nombre": "S3D",
        "constructor": s3d,
        "shape": (1, 3, 16, 224, 224)
    },
    {
        "nombre": "MViT-V2-S",
        "constructor": mvit_v2_s,
        "shape": (1, 3, 16, 224, 224)
    }
]


resultados_forma = []


for prueba in pruebas_entrada:

    nombre = prueba["nombre"]
    constructor = prueba["constructor"]
    shape = prueba["shape"]

    print(f"\n=== {nombre} ===")
    print("Entrada esperada:", shape)

    try:
        modelo = constructor(
            weights=None
        )

        modelo.eval()

        entrada = torch.zeros(
            shape,
            dtype=torch.float32
        )

        with torch.inference_mode():
            salida = modelo(
                entrada
            )

        print(
            "Salida:",
            tuple(salida.shape)
        )

        estado = "OK"

    except Exception as exc:
        print(
            "Error:",
            type(exc).__name__
        )

        print(
            "Detalle:",
            str(exc)
        )

        estado = "ERROR"

    resultados_forma.append(
        {
            "modelo": nombre,
            "entrada": str(shape),
            "estado": estado
        }
    )

    del modelo
    del entrada

    if "salida" in locals():
        del salida

    gc.collect()


df_resultados_forma = pd.DataFrame(
    resultados_forma
)

print(
    "\n=== RESUMEN DE COMPATIBILIDAD ==="
)

display(
    df_resultados_forma
)

##### Conclusiones de 8.2.2

Las cinco arquitecturas RGB y sus pesos preentrenados se encuentran disponibles en la versión de Torchvision utilizada por el proyecto.

Las transformaciones asociadas a los pesos confirman dos configuraciones espaciales principales. R3D-18, MC3-18 y R(2+1)D-18 utilizan entradas de 112 × 112 píxeles, mientras que S3D y MViT-V2-S utilizan entradas de 224 × 224 píxeles. El muestreo temporal se mantiene constante en 16 frames para todos los modelos.

La prueba estructural confirmó que las cinco arquitecturas aceptan correctamente las formas de entrada propuestas:

| Modelo | Forma de entrada | Estado |
|---|---|---|
| R3D-18 | `(1, 3, 16, 112, 112)` | Correcto |
| MC3-18 | `(1, 3, 16, 112, 112)` | Correcto |
| R(2+1)D-18 | `(1, 3, 16, 112, 112)` | Correcto |
| S3D | `(1, 3, 16, 224, 224)` | Correcto |
| MViT-V2-S | `(1, 3, 16, 224, 224)` | Correcto |

En esta etapa las arquitecturas conservan sus clasificadores originales de 400 clases correspondientes a Kinetics-400. La adaptación a las dos clases del proyecto se realizará posteriormente.

Con estas verificaciones queda validada la compatibilidad entre el muestreo temporal común, el preprocesamiento específico de cada arquitectura y las formas de entrada requeridas por los cinco modelos RGB.

#### 8.2.3 Implementación del dataset RGB común

Para realizar una comparación consistente entre las cinco arquitecturas se implementa una clase de dataset común encargada de la lectura y el muestreo temporal de los clips.

Cada video se representa mediante 16 frames seleccionados uniformemente a lo largo de su duración. Los frames se leen en formato RGB y se conservan inicialmente sin aplicar una resolución espacial específica.

Esta separación permite mantener constante la unidad temporal de análisis y aplicar posteriormente la transformación requerida por cada arquitectura.

La implementación divide el procesamiento en tres etapas:

1. selección uniforme de 16 frames de cada clip
2. aplicación del preprocesamiento específico del modelo
3. asociación del tensor resultante con su etiqueta binaria

A partir de esta clase se construirán las particiones `train`, `validation` y `test` para cada arquitectura utilizando exactamente los mismos registros del dataset v2.

Antes de continuar se verificará que las muestras producidas tengan la forma, tipo de dato y valores numéricos esperados para cada modelo.

In [ ]:
# ============================================================
# Dataset RGB común para los modelos del Avance 4
# ============================================================

import cv2

from torch.utils.data import Dataset


NUM_FRAMES = 16


def cargar_frames_rgb(
    video_path,
    num_frames=NUM_FRAMES
):
    """
    Muestrea frames distribuidos uniformemente a lo largo
    de un clip y devuelve un tensor con forma (T, C, H, W).
    """

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise RuntimeError(
            "No se pudo abrir el clip: "
            f"{video_path.name}"
        )

    n_frames = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    if n_frames < num_frames:
        cap.release()

        raise RuntimeError(
            f"El clip {video_path.name} contiene "
            f"{n_frames} frames y requiere al menos "
            f"{num_frames}."
        )

    indices = np.linspace(
        0,
        n_frames - 1,
        num_frames
    )

    indices = np.round(
        indices
    ).astype(int)

    frames = []

    for idx in indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(idx)
        )

        ret, frame = cap.read()

        if not ret:
            cap.release()

            raise RuntimeError(
                f"No se pudo leer el frame {idx} "
                f"del clip {video_path.name}"
            )

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        frame = torch.from_numpy(
            frame.copy()
        )

        frame = frame.permute(
            2,
            0,
            1
        )

        frames.append(
            frame
        )

    cap.release()

    clip = torch.stack(
        frames,
        dim=0
    )

    return clip


class VideoDatasetRGB(Dataset):
    """
    Dataset común para clasificación binaria de clips RGB.

    La transformación espacial y la normalización se reciben
    como argumento para adaptarse a cada arquitectura.
    """

    def __init__(
        self,
        dataframe,
        clips_dir,
        transform
    ):
        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.clips_dir = Path(
            clips_dir
        )

        self.transform = transform

    def __len__(self):
        return len(
            self.df
        )

    def __getitem__(
        self,
        idx
    ):
        fila = self.df.iloc[
            idx
        ]

        video_path = (
            self.clips_dir
            / fila["nombre_clip"]
        )

        if not video_path.exists():
            raise FileNotFoundError(
                "No se encontró el clip requerido: "
                f"{video_path.name}"
            )

        clip = cargar_frames_rgb(
            video_path
        )

        if self.transform is not None:
            clip = self.transform(
                clip
            )

        etiqueta = torch.tensor(
            int(
                fila["clase_binaria"]
            ),
            dtype=torch.long
        )

        return clip, etiqueta


print(
    "Dataset RGB común definido correctamente."
)

In [ ]:
# ============================================================
# Transformaciones específicas por arquitectura
# ============================================================

TRANSFORMS_RGB = {
    "R3D-18": (
        video_models
        .R3D_18_Weights
        .DEFAULT
        .transforms()
    ),
    "MC3-18": (
        video_models
        .MC3_18_Weights
        .DEFAULT
        .transforms()
    ),
    "R(2+1)D-18": (
        video_models
        .R2Plus1D_18_Weights
        .DEFAULT
        .transforms()
    ),
    "S3D": (
        video_models
        .S3D_Weights
        .DEFAULT
        .transforms()
    ),
    "MViT-V2-S": (
        video_models
        .MViT_V2_S_Weights
        .DEFAULT
        .transforms()
    )
}


nombre_clip_prueba = (
    df_train_v2
    .iloc[0]
    ["nombre_clip"]
)

video_prueba = (
    CLIPS_DIR
    / nombre_clip_prueba
)

clip_raw = cargar_frames_rgb(
    video_prueba
)

print("=== CLIP CRUDO ===")
print("Clip:", nombre_clip_prueba)
print("Forma:", tuple(clip_raw.shape))
print("Tipo:", clip_raw.dtype)


resultados_transformacion = []

for nombre_modelo, transform in TRANSFORMS_RGB.items():

    clip_transformado = transform(
        clip_raw
    )

    resultados_transformacion.append(
        {
            "modelo": nombre_modelo,
            "forma": str(
                tuple(
                    clip_transformado.shape
                )
            ),
            "tipo": str(
                clip_transformado.dtype
            ),
            "finitos": bool(
                torch.isfinite(
                    clip_transformado
                )
                .all()
                .item()
            )
        }
    )


df_transformaciones = pd.DataFrame(
    resultados_transformacion
)

print(
    "\n=== RESULTADO DE TRANSFORMACIONES ==="
)

display(
    df_transformaciones
)

In [ ]:
# ============================================================
# Construcción de datasets por arquitectura
# ============================================================

datasets_rgb = {}

for nombre_modelo, transform in TRANSFORMS_RGB.items():

    datasets_rgb[nombre_modelo] = {
        "train": VideoDatasetRGB(
            df_train_v2,
            CLIPS_DIR,
            transform
        ),
        "val": VideoDatasetRGB(
            df_val_v2,
            CLIPS_DIR,
            transform
        ),
        "test": VideoDatasetRGB(
            df_test_modelado_v2,
            CLIPS_DIR,
            transform
        )
    }


resultados_datasets = []

for nombre_modelo, particiones in datasets_rgb.items():

    clip_prueba, etiqueta_prueba = (
        particiones["train"][0]
    )

    resultados_datasets.append(
        {
            "modelo": nombre_modelo,
            "train": len(
                particiones["train"]
            ),
            "val": len(
                particiones["val"]
            ),
            "test": len(
                particiones["test"]
            ),
            "forma_muestra_train": str(
                tuple(
                    clip_prueba.shape
                )
            ),
            "tipo_tensor": str(
                clip_prueba.dtype
            ),
            "etiqueta_muestra": int(
                etiqueta_prueba.item()
            ),
            "finitos": bool(
                torch.isfinite(
                    clip_prueba
                )
                .all()
                .item()
            )
        }
    )


df_datasets_rgb = pd.DataFrame(
    resultados_datasets
)


assert (
    df_datasets_rgb["train"]
    .eq(702)
    .all()
)

assert (
    df_datasets_rgb["val"]
    .eq(136)
    .all()
)

assert (
    df_datasets_rgb["test"]
    .eq(136)
    .all()
)

assert (
    df_datasets_rgb["finitos"]
    .all()
)


print(
    "=== DATASETS RGB VERIFICADOS ==="
)

display(
    df_datasets_rgb
)

##### Conclusiones de 8.2.3

La implementación del dataset RGB común quedó validada para las cinco arquitecturas.

La lectura de video utiliza el mismo procedimiento de muestreo temporal para todos los modelos. Cada clip aporta 16 frames distribuidos uniformemente a lo largo de su duración y posteriormente se aplica la transformación específica correspondiente a la arquitectura.

Las cinco variantes del dataset utilizan exactamente las mismas observaciones:

| Partición | Clips |
|---|---:|
| Train | 702 |
| Validation | 136 |
| Test | 136 |

Las transformaciones producen tensores `torch.float32` con valores finitos. R3D-18, MC3-18 y R(2+1)D-18 generan muestras con forma `(3, 16, 112, 112)`, mientras que S3D y MViT-V2-S generan muestras con forma `(3, 16, 224, 224)`.

La separación entre lectura temporal y transformación espacial permite reutilizar una misma implementación de dataset para las cinco arquitecturas y mantener constantes las observaciones, etiquetas y selección temporal de frames.

Con esta verificación queda preparada la entrada de datos RGB para las siguientes etapas de adaptación y entrenamiento de los modelos.

#### 8.2.4 Adaptación binaria y política inicial de fine-tuning

Las arquitecturas seleccionadas fueron preentrenadas para clasificación sobre Kinetics-400, por lo que sus capas finales producen originalmente 400 salidas.

Antes del entrenamiento para el problema de ocultamiento se inspeccionará la estructura final de cada arquitectura y se sustituirá su clasificador original por una salida de dos clases:

- `0`: `sin_ocultamiento`
- `1`: `ocultamiento`

La adaptación debe respetar la estructura propia de cada modelo. R3D-18, MC3-18 y R(2+1)D-18 utilizan una capa lineal final. S3D emplea una convolución tridimensional como parte de su clasificador y MViT-V2-S utiliza una cabeza basada en una capa lineal posterior a Dropout.

Después de adaptar las salidas se inspeccionarán los bloques finales de representación para definir qué parte de cada backbone permanecerá entrenable durante la comparación inicial.

La política de fine-tuning buscará conservar la mayor parte de las representaciones aprendidas durante el preentrenamiento y, al mismo tiempo, permitir que las capas de mayor nivel se adapten al dominio de retail.

Debido a que las arquitecturas tienen estructuras y cantidades de parámetros diferentes, la comparación no exigirá exactamente el mismo número de parámetros entrenables. Se buscará una profundidad de ajuste aproximadamente comparable y se verificará el porcentaje de parámetros actualizados antes de fijar la configuración definitiva.

In [ ]:
# ============================================================
# Inspección de las capas clasificadoras finales
# ============================================================

constructores_rgb = {
    "R3D-18": r3d_18,
    "MC3-18": mc3_18,
    "R(2+1)D-18": r2plus1d_18,
    "S3D": s3d,
    "MViT-V2-S": mvit_v2_s
}


print("=== CLASIFICADORES ORIGINALES ===")


for nombre_modelo, constructor in constructores_rgb.items():

    modelo = constructor(
        weights=None
    )

    print(f"\n{nombre_modelo}")

    if hasattr(modelo, "fc"):
        print("Atributo final: fc")
        print(modelo.fc)

    elif hasattr(modelo, "classifier"):
        print("Atributo final: classifier")
        print(modelo.classifier)

    elif hasattr(modelo, "head"):
        print("Atributo final: head")
        print(modelo.head)

    else:
        print(
            "Clasificador final no identificado"
        )

    del modelo

    gc.collect()

In [ ]:
# ============================================================
# Adaptación de arquitecturas RGB a clasificación binaria
# ============================================================

import torch.nn as nn


PESOS_RGB = {
    "R3D-18": video_models.R3D_18_Weights.DEFAULT,
    "MC3-18": video_models.MC3_18_Weights.DEFAULT,
    "R(2+1)D-18": video_models.R2Plus1D_18_Weights.DEFAULT,
    "S3D": video_models.S3D_Weights.DEFAULT,
    "MViT-V2-S": video_models.MViT_V2_S_Weights.DEFAULT
}


def crear_modelo_rgb(
    nombre_modelo,
    num_classes=2,
    pretrained=True
):
    """
    Crea una arquitectura RGB y sustituye su clasificador
    original por una salida binaria.
    """

    if nombre_modelo not in constructores_rgb:
        raise ValueError(
            f"Modelo no reconocido: {nombre_modelo}"
        )

    constructor = constructores_rgb[
        nombre_modelo
    ]

    pesos = (
        PESOS_RGB[nombre_modelo]
        if pretrained
        else None
    )

    modelo = constructor(
        weights=pesos
    )

    if nombre_modelo in [
        "R3D-18",
        "MC3-18",
        "R(2+1)D-18"
    ]:
        in_features = modelo.fc.in_features

        modelo.fc = nn.Linear(
            in_features,
            num_classes
        )

    elif nombre_modelo == "S3D":
        in_channels = (
            modelo
            .classifier[1]
            .in_channels
        )

        modelo.classifier[1] = nn.Conv3d(
            in_channels=in_channels,
            out_channels=num_classes,
            kernel_size=1
        )

    elif nombre_modelo == "MViT-V2-S":
        in_features = (
            modelo
            .head[1]
            .in_features
        )

        modelo.head[1] = nn.Linear(
            in_features,
            num_classes
        )

    return modelo


# ------------------------------------------------------------
# Verificación estructural
# ------------------------------------------------------------

resultados_binarios = []


for nombre_modelo in constructores_rgb:

    modelo = crear_modelo_rgb(
        nombre_modelo,
        num_classes=2,
        pretrained=False
    )

    if hasattr(modelo, "fc"):
        clasificador = modelo.fc

    elif hasattr(modelo, "classifier"):
        clasificador = modelo.classifier

    else:
        clasificador = modelo.head

    parametros_totales = sum(
        p.numel()
        for p in modelo.parameters()
    )

    resultados_binarios.append(
        {
            "modelo": nombre_modelo,
            "clasificador_binario": str(
                clasificador
            ),
            "parametros_totales": parametros_totales
        }
    )

    del modelo

    gc.collect()


df_modelos_binarios = pd.DataFrame(
    resultados_binarios
)


print(
    "=== MODELOS ADAPTADOS A DOS CLASES ==="
)

display(
    df_modelos_binarios
)

In [ ]:
# ============================================================
# Inspección de bloques finales de cada arquitectura
# ============================================================

print(
    "=== BLOQUES FINALES DE LAS ARQUITECTURAS RGB ==="
)


for nombre_modelo in constructores_rgb:

    modelo = crear_modelo_rgb(
        nombre_modelo,
        num_classes=2,
        pretrained=False
    )

    print(
        f"\n{'=' * 60}"
    )
    print(
        nombre_modelo
    )
    print(
        "=" * 60
    )

    print(
        "\nMódulos principales:"
    )

    for nombre, modulo in modelo.named_children():
        print(
            f"  {nombre}: "
            f"{modulo.__class__.__name__}"
        )

    if nombre_modelo in [
        "R3D-18",
        "MC3-18",
        "R(2+1)D-18"
    ]:

        print(
            "\nBloque final residual:"
        )

        print(
            "layer4 contiene:",
            len(modelo.layer4),
            "subbloques"
        )

        for idx, bloque in enumerate(
            modelo.layer4
        ):
            print(
                f"  layer4[{idx}]: "
                f"{bloque.__class__.__name__}"
            )

    elif nombre_modelo == "S3D":

        print(
            "\nParte final de features:"
        )

        print(
            "Número de módulos:",
            len(modelo.features)
        )

        inicio = max(
            0,
            len(modelo.features) - 4
        )

        for idx in range(
            inicio,
            len(modelo.features)
        ):
            print(
                f"  features[{idx}]: "
                f"{modelo.features[idx].__class__.__name__}"
            )

        print(
            "\nClasificador:"
        )

        print(
            modelo.classifier
        )

    elif nombre_modelo == "MViT-V2-S":

        print(
            "\nBloques Transformer:"
        )

        print(
            "Número de bloques:",
            len(modelo.blocks)
        )

        inicio = max(
            0,
            len(modelo.blocks) - 3
        )

        for idx in range(
            inicio,
            len(modelo.blocks)
        ):
            print(
                f"  blocks[{idx}]: "
                f"{modelo.blocks[idx].__class__.__name__}"
            )

        print(
            "\nNormalización final:"
        )

        print(
            modelo.norm
        )

        print(
            "\nHead:"
        )

        print(
            modelo.head
        )

    del modelo

    gc.collect()

In [ ]:
# ============================================================
# Política inicial común de fine-tuning
# ============================================================

def configurar_parametros_entrenables(
    modelo,
    nombre_modelo
):
    """
    Congela toda la arquitectura y descongela únicamente
    el último bloque de representación y el clasificador.
    """

    for parametro in modelo.parameters():
        parametro.requires_grad = False

    if nombre_modelo in [
        "R3D-18",
        "MC3-18",
        "R(2+1)D-18"
    ]:

        for parametro in (
            modelo
            .layer4[1]
            .parameters()
        ):
            parametro.requires_grad = True

        for parametro in (
            modelo
            .fc
            .parameters()
        ):
            parametro.requires_grad = True

    elif nombre_modelo == "S3D":

        for parametro in (
            modelo
            .features[15]
            .parameters()
        ):
            parametro.requires_grad = True

        for parametro in (
            modelo
            .classifier
            .parameters()
        ):
            parametro.requires_grad = True

    elif nombre_modelo == "MViT-V2-S":

        for parametro in (
            modelo
            .blocks[15]
            .parameters()
        ):
            parametro.requires_grad = True

        for parametro in (
            modelo
            .norm
            .parameters()
        ):
            parametro.requires_grad = True

        for parametro in (
            modelo
            .head
            .parameters()
        ):
            parametro.requires_grad = True

    else:
        raise ValueError(
            f"Modelo no reconocido: {nombre_modelo}"
        )

    return modelo


resultados_finetuning = []


for nombre_modelo in constructores_rgb:

    modelo = crear_modelo_rgb(
        nombre_modelo,
        num_classes=2,
        pretrained=False
    )

    modelo = configurar_parametros_entrenables(
        modelo,
        nombre_modelo
    )

    parametros_totales = sum(
        p.numel()
        for p in modelo.parameters()
    )

    parametros_entrenables = sum(
        p.numel()
        for p in modelo.parameters()
        if p.requires_grad
    )

    porcentaje_entrenable = (
        100
        * parametros_entrenables
        / parametros_totales
    )

    tensores_entrenables = [
        nombre
        for nombre, parametro
        in modelo.named_parameters()
        if parametro.requires_grad
    ]

    resultados_finetuning.append(
        {
            "modelo": nombre_modelo,
            "parametros_totales": parametros_totales,
            "parametros_entrenables": parametros_entrenables,
            "porcentaje_entrenable": round(
                porcentaje_entrenable,
                2
            ),
            "tensores_entrenables": len(
                tensores_entrenables
            )
        }
    )

    del modelo

    gc.collect()


df_finetuning_inicial = pd.DataFrame(
    resultados_finetuning
)


print(
    "=== POLÍTICA INICIAL DE FINE-TUNING ==="
)

display(
    df_finetuning_inicial
)

In [ ]:
# ============================================================
# Comparación de profundidad de fine-tuning
# para S3D y MViT-V2-S
# ============================================================

def contar_parametros_entrenables(
    modelo
):
    totales = sum(
        p.numel()
        for p in modelo.parameters()
    )

    entrenables = sum(
        p.numel()
        for p in modelo.parameters()
        if p.requires_grad
    )

    porcentaje = (
        100
        * entrenables
        / totales
    )

    return (
        totales,
        entrenables,
        porcentaje
    )


resultados_profundidad = []


# ------------------------------------------------------------
# S3D con un bloque final
# ------------------------------------------------------------

modelo = crear_modelo_rgb(
    "S3D",
    num_classes=2,
    pretrained=False
)

modelo = configurar_parametros_entrenables(
    modelo,
    "S3D"
)

totales, entrenables, porcentaje = (
    contar_parametros_entrenables(
        modelo
    )
)

resultados_profundidad.append(
    {
        "modelo": "S3D",
        "politica": "1 bloque",
        "parametros_entrenables": entrenables,
        "porcentaje_entrenable": round(
            porcentaje,
            2
        )
    }
)

del modelo
gc.collect()


# ------------------------------------------------------------
# S3D con dos bloques finales
# ------------------------------------------------------------

modelo = crear_modelo_rgb(
    "S3D",
    num_classes=2,
    pretrained=False
)

for parametro in modelo.parameters():
    parametro.requires_grad = False

for idx in [14, 15]:
    for parametro in (
        modelo
        .features[idx]
        .parameters()
    ):
        parametro.requires_grad = True

for parametro in (
    modelo
    .classifier
    .parameters()
):
    parametro.requires_grad = True

totales, entrenables, porcentaje = (
    contar_parametros_entrenables(
        modelo
    )
)

resultados_profundidad.append(
    {
        "modelo": "S3D",
        "politica": "2 bloques",
        "parametros_entrenables": entrenables,
        "porcentaje_entrenable": round(
            porcentaje,
            2
        )
    }
)

del modelo
gc.collect()


# ------------------------------------------------------------
# MViT con un bloque final
# ------------------------------------------------------------

modelo = crear_modelo_rgb(
    "MViT-V2-S",
    num_classes=2,
    pretrained=False
)

modelo = configurar_parametros_entrenables(
    modelo,
    "MViT-V2-S"
)

totales, entrenables, porcentaje = (
    contar_parametros_entrenables(
        modelo
    )
)

resultados_profundidad.append(
    {
        "modelo": "MViT-V2-S",
        "politica": "1 bloque",
        "parametros_entrenables": entrenables,
        "porcentaje_entrenable": round(
            porcentaje,
            2
        )
    }
)

del modelo
gc.collect()


# ------------------------------------------------------------
# MViT con dos bloques finales
# ------------------------------------------------------------

modelo = crear_modelo_rgb(
    "MViT-V2-S",
    num_classes=2,
    pretrained=False
)

for parametro in modelo.parameters():
    parametro.requires_grad = False

for idx in [14, 15]:
    for parametro in (
        modelo
        .blocks[idx]
        .parameters()
    ):
        parametro.requires_grad = True

for parametro in (
    modelo
    .norm
    .parameters()
):
    parametro.requires_grad = True

for parametro in (
    modelo
    .head
    .parameters()
):
    parametro.requires_grad = True

totales, entrenables, porcentaje = (
    contar_parametros_entrenables(
        modelo
    )
)

resultados_profundidad.append(
    {
        "modelo": "MViT-V2-S",
        "politica": "2 bloques",
        "parametros_entrenables": entrenables,
        "porcentaje_entrenable": round(
            porcentaje,
            2
        )
    }
)

del modelo
gc.collect()


df_profundidad_finetuning = pd.DataFrame(
    resultados_profundidad
)


print(
    "=== COMPARACIÓN DE PROFUNDIDAD DE FINE-TUNING ==="
)

display(
    df_profundidad_finetuning
)

In [ ]:
# ============================================================
# Política definitiva de fine-tuning para comparación inicial
# ============================================================

def configurar_parametros_entrenables(
    modelo,
    nombre_modelo
):
    """
    Congela el backbone y habilita una porción final
    aproximadamente comparable entre arquitecturas.
    """

    for parametro in modelo.parameters():
        parametro.requires_grad = False

    if nombre_modelo in [
        "R3D-18",
        "MC3-18",
        "R(2+1)D-18"
    ]:

        for parametro in (
            modelo
            .layer4[1]
            .parameters()
        ):
            parametro.requires_grad = True

        for parametro in (
            modelo
            .fc
            .parameters()
        ):
            parametro.requires_grad = True

    elif nombre_modelo == "S3D":

        for idx in [14, 15]:
            for parametro in (
                modelo
                .features[idx]
                .parameters()
            ):
                parametro.requires_grad = True

        for parametro in (
            modelo
            .classifier
            .parameters()
        ):
            parametro.requires_grad = True

    elif nombre_modelo == "MViT-V2-S":

        for idx in [14, 15]:
            for parametro in (
                modelo
                .blocks[idx]
                .parameters()
            ):
                parametro.requires_grad = True

        for parametro in (
            modelo
            .norm
            .parameters()
        ):
            parametro.requires_grad = True

        for parametro in (
            modelo
            .head
            .parameters()
        ):
            parametro.requires_grad = True

    else:
        raise ValueError(
            f"Modelo no reconocido: {nombre_modelo}"
        )

    return modelo


resultados_politica_final = []


for nombre_modelo in constructores_rgb:

    modelo = crear_modelo_rgb(
        nombre_modelo,
        num_classes=2,
        pretrained=False
    )

    modelo = configurar_parametros_entrenables(
        modelo,
        nombre_modelo
    )

    parametros_totales = sum(
        p.numel()
        for p in modelo.parameters()
    )

    parametros_entrenables = sum(
        p.numel()
        for p in modelo.parameters()
        if p.requires_grad
    )

    porcentaje_entrenable = (
        100
        * parametros_entrenables
        / parametros_totales
    )

    resultados_politica_final.append(
        {
            "modelo": nombre_modelo,
            "parametros_totales": parametros_totales,
            "parametros_entrenables": parametros_entrenables,
            "porcentaje_entrenable": round(
                porcentaje_entrenable,
                2
            )
        }
    )

    del modelo
    gc.collect()


df_politica_final = pd.DataFrame(
    resultados_politica_final
)


print(
    "=== POLÍTICA DEFINITIVA DE FINE-TUNING INICIAL ==="
)

display(
    df_politica_final
)

##### Conclusiones de 8.2.4

Las cinco arquitecturas fueron adaptadas correctamente al problema de clasificación binaria mediante la sustitución de sus clasificadores originales de Kinetics-400 por salidas correspondientes a `sin_ocultamiento` y `ocultamiento`.

La estructura del clasificador se conservó de acuerdo con el diseño de cada arquitectura. R3D-18, MC3-18 y R(2+1)D-18 utilizan una capa lineal final. S3D conserva su esquema con Dropout y convolución tridimensional, mientras que MViT-V2-S mantiene Dropout seguido de una capa lineal.

La inspección de los bloques finales mostró que utilizar una única etapa entrenable en S3D y MViT-V2-S producía una proporción de parámetros ajustables considerablemente menor que en las arquitecturas residuales. Por esta razón se evaluó una profundidad adicional antes de fijar la política definitiva.

La configuración seleccionada para la comparación inicial es:

| Modelo | Bloques entrenables | Parámetros entrenables | Porcentaje entrenable |
|---|---|---:|---:|
| R3D-18 | `layer4[1]` y `fc` | 14,158,850 | 42.69 % |
| MC3-18 | `layer4[1]` y `fc` | 4,721,666 | 41.09 % |
| R(2+1)D-18 | `layer4[1]` y `fc` | 14,163,458 | 45.25 % |
| S3D | `features[14]`, `features[15]` y `classifier` | 3,177,826 | 40.16 % |
| MViT-V2-S | `blocks[14]`, `blocks[15]`, `norm` y `head` | 13,616,258 | 39.78 % |

La proporción entrenable queda así dentro de un intervalo aproximado de 40 % a 45 % para las cinco arquitecturas.

Este criterio no implica que porcentajes similares representen exactamente la misma capacidad de adaptación entre modelos con estructuras distintas. Su propósito es evitar diferencias extremas en la profundidad del fine-tuning durante la comparación inicial.

Con esta política queda definida la parte de cada arquitectura que será actualizada durante el entrenamiento. El siguiente paso consiste en verificar si esta configuración puede ejecutarse con un tamaño de batch común bajo la memoria disponible de la GPU.

#### 8.2.5 Validación de la configuración computacional

Antes de iniciar los entrenamientos completos se valida empíricamente que la configuración de fine-tuning definida para cada arquitectura pueda ejecutarse con los recursos computacionales disponibles.

La prueba reproduce un paso real de entrenamiento mediante propagación hacia adelante, cálculo de la pérdida, propagación hacia atrás y actualización de parámetros con AdamW utilizando precisión mixta.

El objetivo principal es determinar si las cinco arquitecturas pueden utilizar un tamaño de batch físico común. Mantener el mismo batch permite reducir una fuente adicional de variación durante la comparación inicial.

Durante la prueba se registran también el pico aproximado de memoria GPU y el tiempo de ejecución por paso. Estos valores se consideran diagnósticos preliminares y no sustituyen las mediciones obtenidas durante los entrenamientos completos.

A partir de los resultados se fijará la configuración definitiva de batch y posteriormente se construirán los `DataLoader` que serán utilizados por los cinco experimentos RGB.

In [ ]:
# ============================================================
# Prueba empírica de batch físico máximo
# ============================================================

import time
from torch.utils.data import DataLoader


if not torch.cuda.is_available():
    raise RuntimeError(
        "Esta prueba requiere CUDA."
    )


def configurar_modo_entrenamiento_rgb(
    modelo,
    nombre_modelo
):
    """
    Mantiene los bloques congelados en evaluación y activa
    únicamente los bloques seleccionados para fine-tuning.
    """

    modelo.eval()

    if nombre_modelo in [
        "R3D-18",
        "MC3-18",
        "R(2+1)D-18"
    ]:
        modelo.layer4[1].train()
        modelo.fc.train()

    elif nombre_modelo == "S3D":
        modelo.features[14].train()
        modelo.features[15].train()
        modelo.classifier.train()

    elif nombre_modelo == "MViT-V2-S":
        modelo.blocks[14].train()
        modelo.blocks[15].train()
        modelo.norm.train()
        modelo.head.train()

    return modelo


CANDIDATOS_BATCH = {
    "R3D-18": [16],
    "MC3-18": [16],
    "R(2+1)D-18": [16],
    "S3D": [16, 8, 4, 2, 1],
    "MViT-V2-S": [16, 8, 4, 2, 1]
}


resultados_batch_gpu = []


for nombre_modelo, candidatos in CANDIDATOS_BATCH.items():

    print(
        f"\n=== {nombre_modelo} ==="
    )

    batch_encontrado = None

    for batch_size in candidatos:

        modelo = None
        optimizador = None
        batch_x = None
        batch_y = None
        salida = None
        perdida = None

        gc.collect()
        torch.cuda.empty_cache()

        try:
            loader_prueba = DataLoader(
                datasets_rgb[nombre_modelo]["train"],
                batch_size=batch_size,
                shuffle=False,
                num_workers=0,
                pin_memory=True
            )

            batch_x, batch_y = next(
                iter(loader_prueba)
            )

            modelo = crear_modelo_rgb(
                nombre_modelo,
                num_classes=2,
                pretrained=False
            )

            modelo = configurar_parametros_entrenables(
                modelo,
                nombre_modelo
            )

            modelo = configurar_modo_entrenamiento_rgb(
                modelo,
                nombre_modelo
            )

            modelo = modelo.to(
                DEVICE
            )

            optimizador = torch.optim.AdamW(
                filter(
                    lambda p: p.requires_grad,
                    modelo.parameters()
                ),
                lr=1e-4,
                weight_decay=1e-4
            )

            criterio = nn.CrossEntropyLoss()

            scaler = torch.amp.GradScaler(
                "cuda"
            )

            batch_x = batch_x.to(
                DEVICE,
                non_blocking=True
            )

            batch_y = batch_y.to(
                DEVICE,
                non_blocking=True
            )

            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()

            inicio = time.perf_counter()

            optimizador.zero_grad(
                set_to_none=True
            )

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                salida = modelo(
                    batch_x
                )

                perdida = criterio(
                    salida,
                    batch_y
                )

            scaler.scale(
                perdida
            ).backward()

            scaler.step(
                optimizador
            )

            scaler.update()

            torch.cuda.synchronize()

            tiempo_s = (
                time.perf_counter()
                - inicio
            )

            memoria_asignada_gb = (
                torch.cuda.max_memory_allocated()
                / 1024**3
            )

            memoria_reservada_gb = (
                torch.cuda.max_memory_reserved()
                / 1024**3
            )

            acumulacion = (
                16 // batch_size
            )

            resultados_batch_gpu.append(
                {
                    "modelo": nombre_modelo,
                    "batch_fisico": batch_size,
                    "estado": "OK",
                    "acumulacion": acumulacion,
                    "batch_efectivo": (
                        batch_size
                        * acumulacion
                    ),
                    "memoria_asignada_gb": round(
                        memoria_asignada_gb,
                        3
                    ),
                    "memoria_reservada_gb": round(
                        memoria_reservada_gb,
                        3
                    ),
                    "tiempo_step_s": round(
                        tiempo_s,
                        3
                    )
                }
            )

            print(
                f"Batch {batch_size}: OK"
            )

            batch_encontrado = batch_size

        except torch.cuda.OutOfMemoryError:

            resultados_batch_gpu.append(
                {
                    "modelo": nombre_modelo,
                    "batch_fisico": batch_size,
                    "estado": "OOM",
                    "acumulacion": None,
                    "batch_efectivo": None,
                    "memoria_asignada_gb": None,
                    "memoria_reservada_gb": None,
                    "tiempo_step_s": None
                }
            )

            print(
                f"Batch {batch_size}: OOM"
            )

        except RuntimeError as exc:

            if "out of memory" in str(exc).lower():

                resultados_batch_gpu.append(
                    {
                        "modelo": nombre_modelo,
                        "batch_fisico": batch_size,
                        "estado": "OOM",
                        "acumulacion": None,
                        "batch_efectivo": None,
                        "memoria_asignada_gb": None,
                        "memoria_reservada_gb": None,
                        "tiempo_step_s": None
                    }
                )

                print(
                    f"Batch {batch_size}: OOM"
                )

            else:
                raise

        finally:

            if modelo is not None:
                del modelo

            if optimizador is not None:
                del optimizador

            if batch_x is not None:
                del batch_x

            if batch_y is not None:
                del batch_y

            if salida is not None:
                del salida

            if perdida is not None:
                del perdida

            gc.collect()
            torch.cuda.empty_cache()

        if batch_encontrado is not None:
            break


df_batch_gpu = pd.DataFrame(
    resultados_batch_gpu
)


print(
    "\n=== PRUEBA DE BATCH FÍSICO ==="
)

display(
    df_batch_gpu
)

In [ ]:
# ============================================================
# Configuración definitiva de batch para comparación RGB
# ============================================================

BATCH_SIZE_RGB = 16

CONFIG_BATCH_RGB = {
    "R3D-18": {
        "batch_size": 16,
        "acumulacion": 1
    },
    "MC3-18": {
        "batch_size": 16,
        "acumulacion": 1
    },
    "R(2+1)D-18": {
        "batch_size": 16,
        "acumulacion": 1
    },
    "S3D": {
        "batch_size": 16,
        "acumulacion": 1
    },
    "MViT-V2-S": {
        "batch_size": 16,
        "acumulacion": 1
    }
}


configuracion_batch_final = []


for nombre_modelo, config in CONFIG_BATCH_RGB.items():

    batch_fisico = config[
        "batch_size"
    ]

    acumulacion = config[
        "acumulacion"
    ]

    configuracion_batch_final.append(
        {
            "modelo": nombre_modelo,
            "batch_fisico": batch_fisico,
            "acumulacion": acumulacion,
            "batch_efectivo": (
                batch_fisico
                * acumulacion
            )
        }
    )


df_batch_final = pd.DataFrame(
    configuracion_batch_final
)


assert (
    df_batch_final[
        "batch_fisico"
    ]
    .eq(16)
    .all()
)

assert (
    df_batch_final[
        "batch_efectivo"
    ]
    .eq(16)
    .all()
)


print(
    "=== CONFIGURACIÓN DEFINITIVA DE BATCH RGB ==="
)

display(
    df_batch_final
)

In [ ]:
# ============================================================
# DataLoaders definitivos para la comparación RGB
# ============================================================

NUM_WORKERS = 0

loaders_rgb = {}


for nombre_modelo, particiones in datasets_rgb.items():

    generator = torch.Generator()
    generator.manual_seed(SEED)

    batch_size = CONFIG_BATCH_RGB[
        nombre_modelo
    ]["batch_size"]

    loaders_rgb[nombre_modelo] = {
        "train": DataLoader(
            particiones["train"],
            batch_size=batch_size,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
            generator=generator
        ),
        "val": DataLoader(
            particiones["val"],
            batch_size=batch_size,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available()
        ),
        "test": DataLoader(
            particiones["test"],
            batch_size=batch_size,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available()
        )
    }


verificacion_loaders_finales = []


for nombre_modelo, particiones in loaders_rgb.items():

    batch_x, batch_y = next(
        iter(
            particiones["train"]
        )
    )

    verificacion_loaders_finales.append(
        {
            "modelo": nombre_modelo,
            "batch_real": int(
                batch_x.shape[0]
            ),
            "forma_batch": str(
                tuple(
                    batch_x.shape
                )
            ),
            "batches_train": len(
                particiones["train"]
            ),
            "batches_val": len(
                particiones["val"]
            ),
            "batches_test": len(
                particiones["test"]
            )
        }
    )


df_loaders_finales = pd.DataFrame(
    verificacion_loaders_finales
)


assert (
    df_loaders_finales[
        "batch_real"
    ]
    .eq(16)
    .all()
)


print(
    "=== DATALOADERS RGB DEFINITIVOS ==="
)

display(
    df_loaders_finales
)

##### Conclusiones de 8.2.5

La configuración computacional quedó validada para realizar la comparación inicial de las cinco arquitecturas RGB bajo un tamaño de batch común.

Todas las arquitecturas completaron correctamente un paso de entrenamiento con batch físico de 16 utilizando precisión mixta. Por esta razón, la configuración definitiva utiliza:

$$
\text{batch físico}=16
$$

$$
\text{batch efectivo}=16
$$

$$
\text{acumulación de gradientes}=1
$$

La prueba preliminar también mostró diferencias relevantes en costo computacional:

| Modelo | Memoria reservada aproximada | Tiempo preliminar por paso |
|---|---:|---:|
| R3D-18 | 0.795 GB | 1.219 s |
| MC3-18 | 0.719 GB | 0.488 s |
| R(2+1)D-18 | 1.443 GB | 0.575 s |
| S3D | 1.705 GB | 0.490 s |
| MViT-V2-S | 5.084 GB | 4.801 s |

MViT-V2-S presentó el mayor consumo de memoria reservada y el mayor tiempo por paso en esta prueba, mientras que MC3-18 presentó el menor consumo de memoria reservada.

Estos valores se consideran únicamente diagnósticos preliminares. Los tiempos de entrenamiento, picos de memoria y velocidades de inferencia utilizados en la comparación final se medirán durante los experimentos completos.

Finalmente, se construyeron y verificaron los `DataLoader` definitivos. Las cinco arquitecturas utilizan batch físico de 16 y conservan las mismas 702 observaciones de entrenamiento, 136 de validación y 136 de prueba.

Con esta validación queda cerrada la preparación computacional previa al entrenamiento de los modelos RGB.

#### 8.2.6 Protocolo común de entrenamiento, evaluación y registro

Una vez definidas las particiones, el preprocesamiento, la política de fine-tuning y la configuración computacional, los cinco modelos RGB se evaluarán mediante un protocolo experimental común.

El objetivo es que las diferencias observadas durante la comparación inicial puedan atribuirse principalmente a la arquitectura evaluada y no a cambios en el procedimiento de entrenamiento o en los criterios de selección.

Para todos los modelos se aplicarán los siguientes principios:

1. Las decisiones de entrenamiento y selección utilizarán exclusivamente `train` y `validation`.
2. `test` permanecerá reservado hasta la evaluación final correspondiente.
3. Los bloques congelados permanecerán en modo de evaluación durante el fine-tuning.
4. Se utilizará precisión mixta para reducir el costo computacional.
5. El mejor checkpoint se seleccionará mediante desempeño sobre validación.
6. Se registrarán las mismas métricas predictivas y computacionales para todas las arquitecturas.

La métrica principal será Average Precision:

$$
AP = \sum_n (R_n - R_{n-1})P_n
$$

donde $P_n$ y $R_n$ representan precisión y recall en los distintos niveles de decisión.

Average Precision se utiliza como criterio principal debido al desbalance de clases y al interés específico en evaluar el desempeño sobre la clase `ocultamiento`.

Como métricas complementarias se registrarán:

- PR-AUC
- ROC-AUC
- EER
- precision
- recall
- specificity
- F1

Las métricas dependientes de umbral se utilizarán para caracterizar el comportamiento operativo del modelo, mientras que AP, PR-AUC, ROC-AUC y EER permiten comparar modelos con menor dependencia de un único punto de decisión.

Además del desempeño predictivo se registrarán variables de costo computacional:

- Número total de parámetros.
- Número y porcentaje de parámetros entrenables.
- Número de épocas ejecutadas.
- Mejor época.
- Tiempo total de entrenamiento.
- Pico de memoria GPU.
- Velocidad de inferencia en clips por segundo.

Cada experimento utilizará una estructura de registro uniforme. Esto permitirá ordenar posteriormente los modelos por Average Precision y analizar simultáneamente desempeño predictivo y costo computacional antes de seleccionar los dos candidatos que continuarán hacia el ajuste de hiperparámetros.

In [ ]:
# ============================================================
# Registro uniforme de experimentos RGB
# ============================================================

COLUMNAS_REGISTRO_RGB = [
    "modelo",
    "parametros_totales",
    "parametros_entrenables",
    "porcentaje_entrenable",
    "batch_fisico",
    "batch_efectivo",
    "epocas_ejecutadas",
    "mejor_epoca",
    "val_average_precision",
    "val_pr_auc",
    "val_roc_auc",
    "val_eer",
    "tiempo_entrenamiento_s",
    "gpu_peak_gb",
    "inferencia_clips_s"
]


def crear_registro_base_rgb(
    modelo,
    nombre_modelo
):
    """
    Genera la estructura base que será completada
    después del entrenamiento de cada arquitectura.
    """

    parametros_totales = sum(
        p.numel()
        for p in modelo.parameters()
    )

    parametros_entrenables = sum(
        p.numel()
        for p in modelo.parameters()
        if p.requires_grad
    )

    porcentaje_entrenable = (
        100
        * parametros_entrenables
        / parametros_totales
    )

    batch_fisico = (
        CONFIG_BATCH_RGB[
            nombre_modelo
        ]["batch_size"]
    )

    acumulacion = (
        CONFIG_BATCH_RGB[
            nombre_modelo
        ]["acumulacion"]
    )

    return {
        "modelo": nombre_modelo,
        "parametros_totales": parametros_totales,
        "parametros_entrenables": parametros_entrenables,
        "porcentaje_entrenable": round(
            porcentaje_entrenable,
            2
        ),
        "batch_fisico": batch_fisico,
        "batch_efectivo": (
            batch_fisico
            * acumulacion
        ),
        "epocas_ejecutadas": None,
        "mejor_epoca": None,
        "val_average_precision": None,
        "val_pr_auc": None,
        "val_roc_auc": None,
        "val_eer": None,
        "tiempo_entrenamiento_s": None,
        "gpu_peak_gb": None,
        "inferencia_clips_s": None
    }


registros_base_rgb = []


for nombre_modelo in constructores_rgb:

    modelo = crear_modelo_rgb(
        nombre_modelo,
        num_classes=2,
        pretrained=False
    )

    modelo = configurar_parametros_entrenables(
        modelo,
        nombre_modelo
    )

    registro = crear_registro_base_rgb(
        modelo,
        nombre_modelo
    )

    registros_base_rgb.append(
        registro
    )

    del modelo
    gc.collect()


df_registro_base_rgb = pd.DataFrame(
    registros_base_rgb,
    columns=COLUMNAS_REGISTRO_RGB
)


print(
    "=== ESTRUCTURA UNIFORME DE REGISTRO RGB ==="
)

display(
    df_registro_base_rgb
)

##### Conclusiones de 8.2.6

El protocolo experimental común quedó definido antes de iniciar el entrenamiento de las arquitecturas RGB.

Los cinco modelos utilizarán las mismas particiones del dataset v2, el mismo muestreo temporal de 16 frames y criterios uniformes para entrenamiento, selección de checkpoints y evaluación. Las diferencias de resolución y normalización se mantienen únicamente cuando forman parte de los requerimientos de los pesos preentrenados de cada arquitectura.

Average Precision será la métrica principal para comparar los modelos durante la etapa de validación. PR-AUC, ROC-AUC y EER se utilizarán como métricas complementarias, junto con precision, recall, specificity y F1 cuando corresponda evaluar un umbral de decisión.

También quedó definida una estructura común de registro que almacenará para cada arquitectura:

- parámetros totales y entrenables
- porcentaje de parámetros entrenables
- batch físico y efectivo
- épocas ejecutadas y mejor época
- métricas de validación
- tiempo total de entrenamiento
- pico de memoria GPU
- velocidad de inferencia

El conjunto de prueba permanecerá fuera del proceso de selección inicial. Los modelos se compararán primero con información de entrenamiento y validación y sólo después de fijar las decisiones de modelado se utilizará test para la evaluación correspondiente.

Con esto queda terminada la preparación común de los experimentos RGB y se puede iniciar la comparación individual de las cinco arquitecturas.

### 8.3 Comparación inicial de modelos RGB

La comparación inicial evalúa las cinco arquitecturas RGB utilizando el protocolo común definido en la sección 8.2.

Todos los modelos utilizarán las mismas particiones de entrenamiento y validación del dataset v2, el mismo muestreo temporal de 16 frames y los criterios comunes de selección y registro. Durante esta etapa el conjunto de prueba permanecerá reservado.

Los modelos se entrenarán y evaluarán individualmente en el siguiente orden:

1. R3D-18
2. MC3-18
3. R(2+1)D-18
4. S3D
5. MViT-V2-S

La comparación inicial utilizará Average Precision sobre validación como métrica principal para ordenar los modelos antes de seleccionar los candidatos que continuarán hacia el ajuste de hiperparámetros.

#### 8.3.1 R3D-18

R3D-18 se utiliza como arquitectura de referencia debido a que fue el baseline desarrollado en el Avance 3.

La configuración del Avance 4 conserva varios elementos metodológicos del baseline anterior:

- pesos preentrenados sobre Kinetics-400
- clasificador final de dos clases
- 16 frames por clip
- entrenamiento de `layer4[1]` y `fc`
- congelamiento del resto del backbone
- selección del modelo mediante datos de validación

Sin embargo, el experimento de esta sección constituye una nueva ejecución y no una reutilización directa del checkpoint del Avance 3.

La diferencia principal se encuentra en el preprocesamiento espacial. En el Avance 3 cada frame completo se redimensionaba directamente a 112 × 112 píxeles antes de aplicar la normalización de Kinetics-400. En el Avance 4 se utiliza la transformación asociada a los pesos preentrenados de Torchvision, que aplica el procedimiento espacial definido para la arquitectura antes de producir la entrada final de 112 × 112 píxeles.

Esta diferencia puede modificar la información espacial observada por el modelo. Por esta razón, utilizar las predicciones históricas del Avance 3 dentro de la comparación arquitectónica del Avance 4 introduciría una condición experimental diferente.

El R3D-18 de esta sección será entrenado nuevamente utilizando el dataset v2 y el pipeline común del Avance 4. Los resultados del Avance 3 se conservarán únicamente como referencia histórica y como base del Experimento 0 de sensibilidad a la calidad del ground truth.

In [ ]:
# ============================================================
# Configuración experimental inicial de R3D-18
# ============================================================

NOMBRE_MODELO_R3D = "R3D-18"

MAX_EPOCHS_RGB = 15

LR_REPRESENTACION_RGB = 1e-4
LR_CLASIFICADOR_RGB = 1e-3
WEIGHT_DECAY_RGB = 1e-4

USAR_AMP_RGB = True

METRICA_CHECKPOINT_RGB = PRIMARY_METRIC

BATCH_R3D = (
    CONFIG_BATCH_RGB[
        NOMBRE_MODELO_R3D
    ]["batch_size"]
)


CONFIG_R3D = {
    "modelo": NOMBRE_MODELO_R3D,
    "max_epochs": MAX_EPOCHS_RGB,
    "lr_representacion": LR_REPRESENTACION_RGB,
    "lr_clasificador": LR_CLASIFICADOR_RGB,
    "weight_decay": WEIGHT_DECAY_RGB,
    "batch_size": BATCH_R3D,
    "precision_mixta": USAR_AMP_RGB,
    "metrica_checkpoint": METRICA_CHECKPOINT_RGB,
    "n_train": len(df_train_v2),
    "n_validation": len(df_val_v2),
    "test_reservado": True
}


assert (
    CONFIG_R3D["batch_size"]
    == 16
)

assert (
    CONFIG_R3D["metrica_checkpoint"]
    == "average_precision"
)

assert (
    CONFIG_R3D["n_train"]
    == 702
)

assert (
    CONFIG_R3D["n_validation"]
    == 136
)


print(
    "=== CONFIGURACIÓN INICIAL R3D-18 ==="
)

display(
    pd.DataFrame(
        [CONFIG_R3D]
    ).T.rename(
        columns={0: "valor"}
    )
)

In [ ]:
# ============================================================
# Construcción y verificación de R3D-18 para Avance 4
# ============================================================

modelo_r3d = crear_modelo_rgb(
    NOMBRE_MODELO_R3D,
    num_classes=2,
    pretrained=True
)

modelo_r3d = configurar_parametros_entrenables(
    modelo_r3d,
    NOMBRE_MODELO_R3D
)

modelo_r3d = configurar_modo_entrenamiento_rgb(
    modelo_r3d,
    NOMBRE_MODELO_R3D
)

modelo_r3d = modelo_r3d.to(
    DEVICE
)


parametros_totales_r3d = sum(
    p.numel()
    for p in modelo_r3d.parameters()
)

parametros_entrenables_r3d = sum(
    p.numel()
    for p in modelo_r3d.parameters()
    if p.requires_grad
)

porcentaje_entrenable_r3d = (
    100
    * parametros_entrenables_r3d
    / parametros_totales_r3d
)


nombres_entrenables_r3d = [
    nombre
    for nombre, parametro
    in modelo_r3d.named_parameters()
    if parametro.requires_grad
]


assert (
    modelo_r3d.fc.out_features
    == 2
)

assert (
    parametros_totales_r3d
    == 33_167_298
)

assert (
    parametros_entrenables_r3d
    == 14_158_850
)

assert all(
    (
        nombre.startswith("layer4.1.")
        or nombre.startswith("fc.")
    )
    for nombre in nombres_entrenables_r3d
)


resumen_modelo_r3d = pd.DataFrame(
    [
        {
            "modelo": NOMBRE_MODELO_R3D,
            "salidas": modelo_r3d.fc.out_features,
            "parametros_totales": parametros_totales_r3d,
            "parametros_entrenables": parametros_entrenables_r3d,
            "porcentaje_entrenable": round(
                porcentaje_entrenable_r3d,
                2
            ),
            "tensores_entrenables": len(
                nombres_entrenables_r3d
            ),
            "dispositivo": str(DEVICE)
        }
    ]
)


print(
    "=== R3D-18 CONFIGURADO PARA AVANCE 4 ==="
)

display(
    resumen_modelo_r3d
)

print(
    "\nParámetros entrenables:"
)

for nombre in nombres_entrenables_r3d:
    print(
        " ",
        nombre
    )

In [ ]:
# ============================================================
# Optimizador, pérdida y forward pass de R3D-18
# ============================================================

import torch.optim as optim


optimizador_r3d = optim.AdamW(
    [
        {
            "params": modelo_r3d.layer4[1].parameters(),
            "lr": LR_REPRESENTACION_RGB,
            "nombre": "representacion"
        },
        {
            "params": modelo_r3d.fc.parameters(),
            "lr": LR_CLASIFICADOR_RGB,
            "nombre": "clasificador"
        }
    ],
    weight_decay=WEIGHT_DECAY_RGB
)


criterio_r3d = nn.CrossEntropyLoss()


amp_habilitado_r3d = (
    USAR_AMP_RGB
    and DEVICE.type == "cuda"
)


scaler_r3d = torch.amp.GradScaler(
    "cuda",
    enabled=amp_habilitado_r3d
)


loader_train_r3d = loaders_rgb[
    NOMBRE_MODELO_R3D
]["train"]


batch_x_r3d, batch_y_r3d = next(
    iter(loader_train_r3d)
)

batch_x_r3d = batch_x_r3d.to(
    DEVICE,
    non_blocking=True
)

batch_y_r3d = batch_y_r3d.to(
    DEVICE,
    non_blocking=True
)


configurar_modo_entrenamiento_rgb(
    modelo_r3d,
    NOMBRE_MODELO_R3D
)


with torch.no_grad():

    with torch.amp.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=amp_habilitado_r3d
    ):

        logits_r3d = modelo_r3d(
            batch_x_r3d
        )

        perdida_prueba_r3d = criterio_r3d(
            logits_r3d,
            batch_y_r3d
        )


assert (
    tuple(batch_x_r3d.shape)
    == (16, 3, 16, 112, 112)
)

assert (
    tuple(logits_r3d.shape)
    == (16, 2)
)

assert torch.isfinite(
    logits_r3d
).all()

assert torch.isfinite(
    perdida_prueba_r3d
)


resumen_optimizador_r3d = pd.DataFrame(
    [
        {
            "grupo": grupo["nombre"],
            "learning_rate": grupo["lr"],
            "weight_decay": grupo["weight_decay"],
            "parametros": sum(
                p.numel()
                for p in grupo["params"]
            )
        }
        for grupo in optimizador_r3d.param_groups
    ]
)


print(
    "=== OPTIMIZADOR R3D-18 ==="
)

display(
    resumen_optimizador_r3d
)


print(
    "\n=== FORWARD PASS ==="
)

print(
    "Forma batch:",
    tuple(batch_x_r3d.shape)
)

print(
    "Forma etiquetas:",
    tuple(batch_y_r3d.shape)
)

print(
    "Forma logits:",
    tuple(logits_r3d.shape)
)

print(
    "Loss de prueba:",
    round(
        float(perdida_prueba_r3d.item()),
        6
    )
)

print(
    "AMP habilitado:",
    amp_habilitado_r3d
)

In [ ]:
# ============================================================
# Funciones comunes de entrenamiento y validación RGB
# ============================================================

def entrenar_una_epoca_rgb(
    modelo,
    nombre_modelo,
    loader,
    criterio,
    optimizador,
    scaler,
    device,
    usar_amp=True
):
    modelo = configurar_modo_entrenamiento_rgb(
        modelo,
        nombre_modelo
    )

    perdida_acumulada = 0.0
    n_observaciones = 0

    for batch_x, batch_y in loader:

        batch_x = batch_x.to(
            device,
            non_blocking=True
        )

        batch_y = batch_y.to(
            device,
            non_blocking=True
        )

        optimizador.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(
                usar_amp
                and device.type == "cuda"
            )
        ):

            logits = modelo(
                batch_x
            )

            perdida = criterio(
                logits,
                batch_y
            )

        scaler.scale(
            perdida
        ).backward()

        scaler.step(
            optimizador
        )

        scaler.update()

        tamano_batch = batch_y.size(0)

        perdida_acumulada += (
            perdida.item()
            * tamano_batch
        )

        n_observaciones += tamano_batch

    perdida_media = (
        perdida_acumulada
        / n_observaciones
    )

    return perdida_media


def evaluar_modelo_rgb(
    modelo,
    loader,
    criterio,
    device,
    usar_amp=True,
    threshold=0.5
):
    modelo.eval()

    perdida_acumulada = 0.0
    n_observaciones = 0

    y_true = []
    y_prob = []

    with torch.inference_mode():

        for batch_x, batch_y in loader:

            batch_x = batch_x.to(
                device,
                non_blocking=True
            )

            batch_y = batch_y.to(
                device,
                non_blocking=True
            )

            with torch.amp.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(
                    usar_amp
                    and device.type == "cuda"
                )
            ):

                logits = modelo(
                    batch_x
                )

                perdida = criterio(
                    logits,
                    batch_y
                )

            probabilidades = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            tamano_batch = batch_y.size(0)

            perdida_acumulada += (
                perdida.item()
                * tamano_batch
            )

            n_observaciones += tamano_batch

            y_true.extend(
                batch_y
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )

            y_prob.extend(
                probabilidades
                .detach()
                .float()
                .cpu()
                .numpy()
                .tolist()
            )

    perdida_media = (
        perdida_acumulada
        / n_observaciones
    )

    metricas = calcular_metricas(
        y_true,
        y_prob,
        threshold=threshold
    )

    return {
        "loss": perdida_media,
        "metricas": metricas,
        "y_true": np.asarray(y_true),
        "y_prob": np.asarray(y_prob)
    }


print(
    "=== FUNCIONES RGB DEFINIDAS ==="
)

print(
    "Entrenamiento:",
    entrenar_una_epoca_rgb.__name__
)

print(
    "Validación:",
    evaluar_modelo_rgb.__name__
)

In [ ]:
# ============================================================
# Prueba controlada de una época completa con R3D-18
# ============================================================

import time


torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


modelo_r3d_prueba = crear_modelo_rgb(
    NOMBRE_MODELO_R3D,
    num_classes=2,
    pretrained=True
)

modelo_r3d_prueba = configurar_parametros_entrenables(
    modelo_r3d_prueba,
    NOMBRE_MODELO_R3D
)

modelo_r3d_prueba = modelo_r3d_prueba.to(
    DEVICE
)


optimizador_r3d_prueba = optim.AdamW(
    [
        {
            "params": modelo_r3d_prueba.layer4[1].parameters(),
            "lr": LR_REPRESENTACION_RGB
        },
        {
            "params": modelo_r3d_prueba.fc.parameters(),
            "lr": LR_CLASIFICADOR_RGB
        }
    ],
    weight_decay=WEIGHT_DECAY_RGB
)


scaler_r3d_prueba = torch.amp.GradScaler(
    "cuda",
    enabled=amp_habilitado_r3d
)


if DEVICE.type == "cuda":
    torch.cuda.synchronize()

inicio_prueba_r3d = time.perf_counter()


loss_train_r3d_prueba = entrenar_una_epoca_rgb(
    modelo=modelo_r3d_prueba,
    nombre_modelo=NOMBRE_MODELO_R3D,
    loader=loaders_rgb[
        NOMBRE_MODELO_R3D
    ]["train"],
    criterio=criterio_r3d,
    optimizador=optimizador_r3d_prueba,
    scaler=scaler_r3d_prueba,
    device=DEVICE,
    usar_amp=USAR_AMP_RGB
)


resultado_val_r3d_prueba = evaluar_modelo_rgb(
    modelo=modelo_r3d_prueba,
    loader=loaders_rgb[
        NOMBRE_MODELO_R3D
    ]["val"],
    criterio=criterio_r3d,
    device=DEVICE,
    usar_amp=USAR_AMP_RGB,
    threshold=0.5
)


if DEVICE.type == "cuda":
    torch.cuda.synchronize()

tiempo_prueba_r3d = (
    time.perf_counter()
    - inicio_prueba_r3d
)


gpu_peak_r3d_prueba = (
    torch.cuda.max_memory_reserved()
    / 1024**3
    if DEVICE.type == "cuda"
    else np.nan
)


metricas_val_r3d_prueba = (
    resultado_val_r3d_prueba[
        "metricas"
    ]
)


resumen_prueba_r3d = pd.DataFrame(
    [
        {
            "train_loss": round(
                loss_train_r3d_prueba,
                4
            ),
            "val_loss": round(
                resultado_val_r3d_prueba[
                    "loss"
                ],
                4
            ),
            "val_AP": round(
                metricas_val_r3d_prueba[
                    "average_precision"
                ],
                4
            ),
            "val_PR_AUC": round(
                metricas_val_r3d_prueba[
                    "pr_auc"
                ],
                4
            ),
            "val_ROC_AUC": round(
                metricas_val_r3d_prueba[
                    "roc_auc"
                ],
                4
            ),
            "val_EER": round(
                metricas_val_r3d_prueba[
                    "eer"
                ],
                4
            ),
            "tiempo_s": round(
                tiempo_prueba_r3d,
                2
            ),
            "gpu_peak_gb": round(
                gpu_peak_r3d_prueba,
                3
            )
        }
    ]
)


print(
    "=== PRUEBA DE UNA ÉPOCA R3D-18 ==="
)

display(
    resumen_prueba_r3d
)


del modelo_r3d_prueba
del optimizador_r3d_prueba
del scaler_r3d_prueba

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Función común para ejecutar un experimento RGB completo
# ============================================================

def entrenar_experimento_rgb(
    modelo,
    nombre_modelo,
    loader_train,
    loader_val,
    criterio,
    optimizador,
    scaler,
    device,
    max_epochs,
    checkpoint_path,
    historial_path,
    experiment_config,
    usar_amp=True
):
    historial = []

    mejor_ap = -np.inf
    mejor_epoca = None


    # --------------------------------------------------------
    # Preparar directorios
    # --------------------------------------------------------

    checkpoint_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    historial_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    # --------------------------------------------------------
    # Preparar medición computacional
    # --------------------------------------------------------

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()


    inicio_total = time.perf_counter()


    # ========================================================
    # Ciclo de entrenamiento
    # ========================================================

    for epoch in range(
        1,
        max_epochs + 1
    ):

        if device.type == "cuda":
            torch.cuda.synchronize()


        inicio_epoca = time.perf_counter()


        # ----------------------------------------------------
        # Entrenamiento
        # ----------------------------------------------------

        train_loss = entrenar_una_epoca_rgb(
            modelo=modelo,
            nombre_modelo=nombre_modelo,
            loader=loader_train,
            criterio=criterio,
            optimizador=optimizador,
            scaler=scaler,
            device=device,
            usar_amp=usar_amp
        )


        # ----------------------------------------------------
        # Validación
        # ----------------------------------------------------

        resultado_val = evaluar_modelo_rgb(
            modelo=modelo,
            loader=loader_val,
            criterio=criterio,
            device=device,
            usar_amp=usar_amp,
            threshold=0.5
        )


        if device.type == "cuda":
            torch.cuda.synchronize()


        tiempo_epoca = (
            time.perf_counter()
            - inicio_epoca
        )


        metricas_val = (
            resultado_val[
                "metricas"
            ]
        )


        # ----------------------------------------------------
        # Registrar época
        # ----------------------------------------------------

        fila = {
            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "val_loss":
                resultado_val[
                    "loss"
                ],

            "val_average_precision":
                metricas_val[
                    "average_precision"
                ],

            "val_pr_auc":
                metricas_val[
                    "pr_auc"
                ],

            "val_roc_auc":
                metricas_val[
                    "roc_auc"
                ],

            "val_eer":
                metricas_val[
                    "eer"
                ],

            "val_precision":
                metricas_val[
                    "precision"
                ],

            "val_recall":
                metricas_val[
                    "recall"
                ],

            "val_specificity":
                metricas_val[
                    "specificity"
                ],

            "val_f1":
                metricas_val[
                    "f1"
                ],

            "tiempo_epoca_s":
                tiempo_epoca
        }


        historial.append(
            fila
        )


        # ----------------------------------------------------
        # Selección del mejor checkpoint por AP
        # ----------------------------------------------------

        ap_actual = (
            metricas_val[
                "average_precision"
            ]
        )


        if ap_actual > mejor_ap:

            mejor_ap = ap_actual
            mejor_epoca = epoch


            checkpoint_temporal = {
                "modelo":
                    nombre_modelo,

                "epoch":
                    epoch,

                "val_average_precision":
                    ap_actual,

                "model_state_dict":
                    modelo.state_dict(),

                "optimizer_state_dict":
                    optimizador.state_dict(),

                "experiment_config":
                    experiment_config
            }


            torch.save(
                checkpoint_temporal,
                checkpoint_path
            )


        # ----------------------------------------------------
        # Seguimiento
        # ----------------------------------------------------

        print(
            f"Época {epoch:02d}/{max_epochs} | "
            f"Train loss {train_loss:.4f} | "
            f"Val loss {resultado_val['loss']:.4f} | "
            f"Val AP {ap_actual:.4f} | "
            f"Val PR-AUC "
            f"{metricas_val['pr_auc']:.4f} | "
            f"{tiempo_epoca:.1f} s"
        )


    # ========================================================
    # Cierre del experimento
    # ========================================================

    if device.type == "cuda":
        torch.cuda.synchronize()


    tiempo_total = (
        time.perf_counter()
        - inicio_total
    )


    gpu_peak_gb = (
        torch.cuda.max_memory_reserved()
        / 1024**3
        if device.type == "cuda"
        else np.nan
    )


    # --------------------------------------------------------
    # Guardar historial completo
    # --------------------------------------------------------

    historial_df = pd.DataFrame(
        historial
    )


    historial_df.to_csv(
        historial_path,
        index=False
    )


    # --------------------------------------------------------
    # Resumen del experimento
    # --------------------------------------------------------

    resumen = {
        "modelo":
            nombre_modelo,

        "epocas_ejecutadas":
            max_epochs,

        "mejor_epoca":
            mejor_epoca,

        "mejor_val_ap":
            mejor_ap,

        "tiempo_entrenamiento_s":
            tiempo_total,

        "gpu_peak_gb":
            gpu_peak_gb
    }


    # --------------------------------------------------------
    # Completar checkpoint definitivo
    # --------------------------------------------------------

    checkpoint_mejor = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )


    checkpoint_mejor[
        "experiment_summary"
    ] = resumen


    torch.save(
        checkpoint_mejor,
        checkpoint_path
    )


    # --------------------------------------------------------
    # Restaurar mejor estado del modelo
    # --------------------------------------------------------

    modelo.load_state_dict(
        checkpoint_mejor[
            "model_state_dict"
        ]
    )


    return {
        "modelo":
            modelo,

        "historial":
            historial_df,

        "resumen":
            resumen,

        "checkpoint_path":
            checkpoint_path,

        "historial_path":
            historial_path
    }


print(
    "=== FUNCIÓN DE EXPERIMENTO RGB DEFINIDA ==="
)

print(
    entrenar_experimento_rgb.__name__
)

In [ ]:
# ============================================================
# Protección contra reentrenamiento de experimentos RGB
# ============================================================

FORCE_RETRAIN_RGB = True


def configuraciones_equivalentes(
    configuracion_actual,
    configuracion_esperada
):
    if (
        not isinstance(
            configuracion_actual,
            dict
        )
        or set(
            configuracion_actual.keys()
        )
        != set(
            configuracion_esperada.keys()
        )
    ):
        return False

    for clave, valor_esperado in (
        configuracion_esperada.items()
    ):

        valor_actual = (
            configuracion_actual[
                clave
            ]
        )

        if isinstance(
            valor_esperado,
            (float, np.floating)
        ):

            if not np.isclose(
                valor_actual,
                valor_esperado
            ):
                return False

        else:

            if (
                valor_actual
                != valor_esperado
            ):
                return False

    return True


def validar_artefactos_rgb(
    checkpoint_path,
    historial_path,
    configuracion_esperada
):
    resultado = {
        "estado": None,
        "motivos": [],
        "checkpoint": None,
        "historial": None
    }


    checkpoint_existe = (
        checkpoint_path.exists()
    )

    historial_existe = (
        historial_path.exists()
    )


    # --------------------------------------------------------
    # No existen artefactos
    # --------------------------------------------------------

    if (
        not checkpoint_existe
        and not historial_existe
    ):
        resultado[
            "estado"
        ] = "ausentes"

        return resultado


    # --------------------------------------------------------
    # Existe sólo uno de los artefactos
    # --------------------------------------------------------

    if (
        checkpoint_existe
        != historial_existe
    ):
        resultado[
            "estado"
        ] = "incompatibles"

        resultado[
            "motivos"
        ].append(
            "Existe solamente uno de los dos artefactos."
        )

        return resultado


    # --------------------------------------------------------
    # Lectura de artefactos
    # --------------------------------------------------------

    try:

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )

        historial = pd.read_csv(
            historial_path
        )

    except Exception as exc:

        resultado[
            "estado"
        ] = "incompatibles"

        resultado[
            "motivos"
        ].append(
            "No fue posible leer los artefactos: "
            f"{type(exc).__name__}"
        )

        return resultado


    resultado[
        "checkpoint"
    ] = checkpoint

    resultado[
        "historial"
    ] = historial


    # --------------------------------------------------------
    # Estructura del checkpoint
    # --------------------------------------------------------

    claves_checkpoint = {
        "modelo",
        "epoch",
        "val_average_precision",
        "model_state_dict",
        "optimizer_state_dict",
        "experiment_config",
        "experiment_summary"
    }


    faltantes_checkpoint = (
        claves_checkpoint
        - set(
            checkpoint.keys()
        )
    )


    if faltantes_checkpoint:

        resultado[
            "motivos"
        ].append(
            "Faltan claves requeridas en el checkpoint: "
            + ", ".join(
                sorted(
                    faltantes_checkpoint
                )
            )
        )


    # --------------------------------------------------------
    # Identidad experimental
    # --------------------------------------------------------

    if (
        "experiment_config"
        in checkpoint
    ):

        if not configuraciones_equivalentes(
            checkpoint[
                "experiment_config"
            ],
            configuracion_esperada
        ):
            resultado[
                "motivos"
            ].append(
                "La configuración almacenada no coincide "
                "con el experimento actual."
            )


    # --------------------------------------------------------
    # Estructura del historial
    # --------------------------------------------------------

    columnas_historial = {
        "epoch",
        "train_loss",
        "val_loss",
        "val_average_precision",
        "val_pr_auc",
        "val_roc_auc",
        "val_eer",
        "val_precision",
        "val_recall",
        "val_specificity",
        "val_f1",
        "tiempo_epoca_s"
    }


    faltantes_historial = (
        columnas_historial
        - set(
            historial.columns
        )
    )


    if faltantes_historial:

        resultado[
            "motivos"
        ].append(
            "Faltan columnas requeridas en el historial: "
            + ", ".join(
                sorted(
                    faltantes_historial
                )
            )
        )


    # --------------------------------------------------------
    # Número y secuencia de épocas
    # --------------------------------------------------------

    max_epochs_esperado = (
        configuracion_esperada[
            "max_epochs"
        ]
    )


    if (
        len(historial)
        != max_epochs_esperado
    ):

        resultado[
            "motivos"
        ].append(
            "El historial no contiene el número esperado "
            "de épocas."
        )


    if (
        "epoch"
        in historial.columns
        and len(historial)
        == max_epochs_esperado
    ):

        epocas_esperadas = list(
            range(
                1,
                max_epochs_esperado + 1
            )
        )

        if (
            historial[
                "epoch"
            ].tolist()
            != epocas_esperadas
        ):

            resultado[
                "motivos"
            ].append(
                "La secuencia de épocas del historial "
                "no es válida."
            )


    # --------------------------------------------------------
    # Coherencia checkpoint-historial
    # --------------------------------------------------------

    columnas_coherencia = {
        "epoch",
        "val_average_precision"
    }


    if (
        columnas_coherencia.issubset(
            historial.columns
        )
        and len(historial) > 0
        and "epoch" in checkpoint
        and "val_average_precision"
        in checkpoint
    ):

        indice_mejor = historial[
            "val_average_precision"
        ].idxmax()


        mejor_epoca_historial = int(
            historial.loc[
                indice_mejor,
                "epoch"
            ]
        )


        mejor_ap_historial = float(
            historial.loc[
                indice_mejor,
                "val_average_precision"
            ]
        )


        if (
            checkpoint[
                "epoch"
            ]
            != mejor_epoca_historial
        ):

            resultado[
                "motivos"
            ].append(
                "La mejor época del checkpoint "
                "no coincide con el historial."
            )


        if not np.isclose(
            checkpoint[
                "val_average_precision"
            ],
            mejor_ap_historial
        ):

            resultado[
                "motivos"
            ].append(
                "El AP almacenado en el checkpoint "
                "no coincide con el historial."
            )


    # --------------------------------------------------------
    # Estado final
    # --------------------------------------------------------

    if (
        len(
            resultado[
                "motivos"
            ]
        )
        == 0
    ):

        resultado[
            "estado"
        ] = "validos"

    else:

        resultado[
            "estado"
        ] = "incompatibles"


    return resultado


print(
    "=== PROTECCIÓN DE EXPERIMENTOS RGB DEFINIDA ==="
)

print(
    "Reentrenamiento forzado:",
    FORCE_RETRAIN_RGB
)

In [ ]:
# ============================================================
# Identidad experimental de R3D-18
# ============================================================

CHECKPOINT_R3D_AVANCE4 = (
    CHECKPOINT_DIR
    / "r3d18_avance4_inicial.pt"
)

HISTORIAL_R3D_AVANCE4 = (
    CHECKPOINT_DIR
    / "r3d18_avance4_inicial_history.csv"
)


DATASET_V2_SHA256 = (
    "b85d6b55e8c32420a4df5a0fb9a10e55"
    "8105beee47f1d4a98ad83d63fcbda21e"
)


CONFIG_IDENTIDAD_R3D = {
    "schema_version": 1,
    "modelo": NOMBRE_MODELO_R3D,
    "seed": SEED,
    "batch_size": BATCH_R3D,
    "max_epochs": MAX_EPOCHS_RGB,
    "lr_representacion": LR_REPRESENTACION_RGB,
    "lr_clasificador": LR_CLASIFICADOR_RGB,
    "weight_decay": WEIGHT_DECAY_RGB,
    "metrica_checkpoint": METRICA_CHECKPOINT_RGB,
    "dataset_v2_sha256": DATASET_V2_SHA256,
    "n_train": len(df_train_v2),
    "n_validation": len(df_val_v2),
    "num_frames": 16,
    "transformacion": repr(
        TRANSFORMS_RGB[
            NOMBRE_MODELO_R3D
        ]
    ),
    "bloques_entrenables": [
        "layer4.1",
        "fc"
    ]
}


assert (
    CONFIG_IDENTIDAD_R3D[
        "dataset_v2_sha256"
    ]
    == DATASET_V2_SHA256
)

assert (
    CONFIG_IDENTIDAD_R3D[
        "n_train"
    ]
    == 702
)

assert (
    CONFIG_IDENTIDAD_R3D[
        "n_validation"
    ]
    == 136
)

assert (
    CONFIG_IDENTIDAD_R3D[
        "batch_size"
    ]
    == 16
)


print(
    "=== IDENTIDAD EXPERIMENTAL R3D-18 ==="
)

display(
    pd.DataFrame(
        [
            {
                "modelo":
                    CONFIG_IDENTIDAD_R3D[
                        "modelo"
                    ],

                "seed":
                    CONFIG_IDENTIDAD_R3D[
                        "seed"
                    ],

                "batch":
                    CONFIG_IDENTIDAD_R3D[
                        "batch_size"
                    ],

                "max_epochs":
                    CONFIG_IDENTIDAD_R3D[
                        "max_epochs"
                    ],

                "lr_representacion":
                    CONFIG_IDENTIDAD_R3D[
                        "lr_representacion"
                    ],

                "lr_clasificador":
                    CONFIG_IDENTIDAD_R3D[
                        "lr_clasificador"
                    ],

                "weight_decay":
                    CONFIG_IDENTIDAD_R3D[
                        "weight_decay"
                    ],

                "metrica":
                    CONFIG_IDENTIDAD_R3D[
                        "metrica_checkpoint"
                    ],

                "n_train":
                    CONFIG_IDENTIDAD_R3D[
                        "n_train"
                    ],

                "n_validation":
                    CONFIG_IDENTIDAD_R3D[
                        "n_validation"
                    ]
            }
        ]
    )
)

print(
    "Dataset v2 SHA-256:",
    DATASET_V2_SHA256
)

print(
    "Checkpoint:",
    CHECKPOINT_R3D_AVANCE4.relative_to(
        REPO_DIR
    )
)

print(
    "Historial:",
    HISTORIAL_R3D_AVANCE4.relative_to(
        REPO_DIR
    )
)

In [ ]:
# ============================================================
# Ejecución protegida del experimento R3D-18
# ============================================================

validacion_r3d = validar_artefactos_rgb(
    checkpoint_path=CHECKPOINT_R3D_AVANCE4,
    historial_path=HISTORIAL_R3D_AVANCE4,
    configuracion_esperada=CONFIG_IDENTIDAD_R3D
)


# ------------------------------------------------------------
# Bloqueo ante artefactos incompatibles
# ------------------------------------------------------------

if (
    validacion_r3d["estado"] == "incompatibles"
    and not FORCE_RETRAIN_RGB
):
    detalle = "\n".join(
        f"- {motivo}"
        for motivo in validacion_r3d["motivos"]
    )

    raise RuntimeError(
        "Los artefactos existentes de R3D-18 son "
        "incompatibles con la configuración actual.\n"
        "El entrenamiento automático fue bloqueado.\n\n"
        f"{detalle}\n\n"
        "Revise la configuración o establezca "
        "FORCE_RETRAIN_RGB=True de forma explícita."
    )


# ============================================================
# Reutilizar experimento existente
# ============================================================

if (
    validacion_r3d["estado"] == "validos"
    and not FORCE_RETRAIN_RGB
):

    checkpoint_r3d = (
        validacion_r3d[
            "checkpoint"
        ]
    )

    historial_r3d = (
        validacion_r3d[
            "historial"
        ].copy()
    )


    modelo_r3d_oficial = crear_modelo_rgb(
        NOMBRE_MODELO_R3D,
        num_classes=2,
        pretrained=True
    )

    modelo_r3d_oficial = (
        configurar_parametros_entrenables(
            modelo_r3d_oficial,
            NOMBRE_MODELO_R3D
        )
    )

    modelo_r3d_oficial.load_state_dict(
        checkpoint_r3d[
            "model_state_dict"
        ]
    )

    modelo_r3d_oficial = (
        modelo_r3d_oficial.to(
            DEVICE
        )
    )


    resultado_r3d = {
        "modelo":
            modelo_r3d_oficial,

        "historial":
            historial_r3d,

        "resumen":
            checkpoint_r3d[
                "experiment_summary"
            ],

        "checkpoint_path":
            CHECKPOINT_R3D_AVANCE4,

        "historial_path":
            HISTORIAL_R3D_AVANCE4
    }


    print(
        "=== R3D-18 REUTILIZADO ==="
    )

    print(
        "Entrenamiento omitido: "
        "los artefactos existentes son válidos."
    )


# ============================================================
# Ejecutar entrenamiento
# ============================================================

else:

    if FORCE_RETRAIN_RGB:

        motivo_entrenamiento = (
            "reentrenamiento solicitado explícitamente"
        )

    else:

        motivo_entrenamiento = (
            "artefactos ausentes"
        )


    print(
        "=== R3D-18 SERÁ ENTRENADO ==="
    )

    print(
        "Motivo:",
        motivo_entrenamiento
    )


    # --------------------------------------------------------
    # Reproducibilidad
    # --------------------------------------------------------

    random.seed(
        SEED
    )

    np.random.seed(
        SEED
    )

    torch.manual_seed(
        SEED
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            SEED
        )

        torch.cuda.empty_cache()


    # --------------------------------------------------------
    # Modelo
    # --------------------------------------------------------

    modelo_r3d_oficial = crear_modelo_rgb(
        NOMBRE_MODELO_R3D,
        num_classes=2,
        pretrained=True
    )

    modelo_r3d_oficial = (
        configurar_parametros_entrenables(
            modelo_r3d_oficial,
            NOMBRE_MODELO_R3D
        )
    )

    modelo_r3d_oficial = (
        modelo_r3d_oficial.to(
            DEVICE
        )
    )


    # --------------------------------------------------------
    # Optimizador
    # --------------------------------------------------------

    optimizador_r3d_oficial = optim.AdamW(
        [
            {
                "params":
                    modelo_r3d_oficial
                    .layer4[1]
                    .parameters(),

                "lr":
                    LR_REPRESENTACION_RGB
            },
            {
                "params":
                    modelo_r3d_oficial
                    .fc
                    .parameters(),

                "lr":
                    LR_CLASIFICADOR_RGB
            }
        ],
        weight_decay=WEIGHT_DECAY_RGB
    )


    # --------------------------------------------------------
    # Pérdida y precisión mixta
    # --------------------------------------------------------

    criterio_r3d_oficial = (
        nn.CrossEntropyLoss()
    )


    scaler_r3d_oficial = (
        torch.amp.GradScaler(
            "cuda",
            enabled=(
                USAR_AMP_RGB
                and DEVICE.type == "cuda"
            )
        )
    )


    # --------------------------------------------------------
    # Entrenamiento completo
    # --------------------------------------------------------

    resultado_r3d = entrenar_experimento_rgb(
        modelo=modelo_r3d_oficial,
        nombre_modelo=NOMBRE_MODELO_R3D,

        loader_train=loaders_rgb[
            NOMBRE_MODELO_R3D
        ]["train"],

        loader_val=loaders_rgb[
            NOMBRE_MODELO_R3D
        ]["val"],

        criterio=criterio_r3d_oficial,
        optimizador=optimizador_r3d_oficial,
        scaler=scaler_r3d_oficial,

        device=DEVICE,

        max_epochs=MAX_EPOCHS_RGB,

        checkpoint_path=(
            CHECKPOINT_R3D_AVANCE4
        ),

        historial_path=(
            HISTORIAL_R3D_AVANCE4
        ),

        experiment_config=(
            CONFIG_IDENTIDAD_R3D
        ),

        usar_amp=USAR_AMP_RGB
    )


    # --------------------------------------------------------
    # Validación obligatoria de los artefactos generados
    # --------------------------------------------------------

    validacion_posterior_r3d = (
        validar_artefactos_rgb(
            checkpoint_path=(
                CHECKPOINT_R3D_AVANCE4
            ),

            historial_path=(
                HISTORIAL_R3D_AVANCE4
            ),

            configuracion_esperada=(
                CONFIG_IDENTIDAD_R3D
            )
        )
    )


    if (
        validacion_posterior_r3d[
            "estado"
        ]
        != "validos"
    ):
        detalle = "\n".join(
            f"- {motivo}"
            for motivo
            in validacion_posterior_r3d[
                "motivos"
            ]
        )

        raise RuntimeError(
            "El experimento terminó, pero los "
            "artefactos generados no superaron "
            "la validación.\n\n"
            f"{detalle}"
        )


# ============================================================
# Resumen común
# ============================================================

resumen_r3d_oficial = pd.DataFrame(
    [
        resultado_r3d[
            "resumen"
        ]
    ]
)


print(
    "\n=== RESUMEN R3D-18 ==="
)

display(
    resumen_r3d_oficial
)

print(
    "Checkpoint:",
    CHECKPOINT_R3D_AVANCE4.relative_to(
        REPO_DIR
    )
)

print(
    "Historial:",
    HISTORIAL_R3D_AVANCE4.relative_to(
        REPO_DIR
    )
)